# 07 · Distribution shift and active sequences

Full-train GRU, event-feature GRU, 2×256 GRU, past-target decoder, adversarial validation (AUC 0.995) and selection of the most active train sequences.

Needs `lib.py` and the `.npy` data written by `00_setup_data_feature_importance`. Comments are in Russian; outputs are kept as the record of each experiment.


In [1]:
import os, shutil, subprocess, time
from google.colab import drive
drive.mount('/content/drive')
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU (Runtime → Change runtime type → T4)')
W = '/content/drive/MyDrive/wnn'
need = ['lib.py', 'gru_full_val_best.pt', 'corr3k_1_best.pt', 'Pb_valid300.npy', 'P_gru6573_valid300.npy',
        'P_corr3k_valid300.npy', 'ens3_submission.zip']
miss = [f for f in need if not os.path.exists(f'{W}/{f}')]
assert not miss, f'на Drive нет {miss}: расшарь папку wnn со старого аккаунта и добавь ярлык в «Мой диск»'

%run -i /content/drive/MyDrive/wnn/lib.py
import pyarrow.parquet as pq
sh = lambda c: print(subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip())
def ok(p, n):
    try: return pq.ParquetFile(p).num_row_groups == n
    except Exception: return False
sh('df -h /content | tail -1')
if not (ok(f'{ROOT}/train.parquet', 10607) and ok(f'{ROOT}/valid.parquet', 1873)):
    t = time.time(); print('качаю стартпак (10–15 минут)...', flush=True)
    sh("cd /content && curl -sL https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | "
       "tar -xz --wildcards '*/baseline/*' '*/utils.py' '*/datasets/*.parquet' 2>/dev/null")
    print(f'скачано за {time.time() - t:.0f} с')
assert ok(f'{ROOT}/train.parquet', 10607) and ok(f'{ROOT}/valid.parquet', 1873), 'parquet битые: перезапусти ячейку'

os.makedirs('/content/data', exist_ok=True); D = '/content/data'
meta = {'seq_ix', 'step_in_seq', 'need_prediction', 'is_scored', 't0', 't1'}
def build16(src, dst, rows):
    pf = pq.ParquetFile(f'{ROOT}/{src}.parquet'); feat = [c for c in pf.schema_arrow.names if c not in meta]; assert len(feat) == NF
    X = np.lib.format.open_memmap(f'{D}/{dst}_X16.npy', 'w+', np.float16, (len(rows), L, NF)); Y = np.zeros((len(rows), L, 2), np.float32); t = time.time()
    for k, i in enumerate(rows):
        tb = pf.read_row_group(int(i), columns=feat + ['t0', 't1'])
        X[k] = np.column_stack([tb[c].to_numpy() for c in feat]).astype(np.float16)
        Y[k] = np.column_stack([tb['t0'].to_numpy(), tb['t1'].to_numpy()])
        if k % 1000 == 0: print(f'  {dst} {k}/{len(rows)} {time.time() - t:.0f} с', flush=True)
    X.flush(); del X; np.save(f'{D}/{dst}_Y.npy', Y)

if not all(os.path.exists(f'{D}/valid300_{s}.npy') for s in 'XYM'): build('valid', 'valid300', 300, 1, True)   # тот же сид
held = set(np.random.default_rng(1).choice(1873, 300, replace=False).tolist())
if not os.path.exists(f'{D}/validrest_Y.npy'): build16('valid', 'validrest', [i for i in range(1873) if i not in held])
os.remove(f'{ROOT}/valid.parquet')                              # больше не нужен, освобождаем место
sh('df -h /content | tail -1'); print('valid готов:', sorted(os.listdir(D)))

Mounted at /content/drive
GPU: Tesla T4
overlay         113G   86M  113G   1% /
качаю стартпак (10–15 минут)...

скачано за 603 с
valid300 0 1 с
  validrest 0/1573 0 с
  validrest 1000/1573 61 с
overlay         113G   37G   76G  33% /
valid готов: ['valid300_M.npy', 'valid300_X.npy', 'valid300_Y.npy', 'validrest_X16.npy', 'validrest_Y.npy']


In [2]:
Xv = np.load('/content/data/valid300_X.npy', mmap_mode='r'); Yv = np.load('/content/data/valid300_Y.npy')
Mv = np.load('/content/data/valid300_M.npy').astype(bool)
famv = fam_of(Yv, verbose=False); hv = np.bincount(famv, minlength=4) / len(famv)
devc = 'cuda' if torch.cuda.is_available() else 'cpu'
net = Net(128).to(devc); net.load_state_dict(torch.load(f'{CKPT_DIR}/gru_full_val_best.pt', map_location=devc)); net.eval()
with torch.no_grad(): o, _ = net(torch.from_numpy(np.ascontiguousarray(Xv[:2, :3000], np.float32)).to(devc))
err = np.abs(o.clamp(-2, 2).cpu().numpy() - np.load(f'{CKPT_DIR}/P_gru6573_valid300.npy', mmap_mode='r')[:2, :3000]).max()
print(f'сверка valid300 с прежним: {err:.1e}'); assert err < 1e-3, 'valid300 собран не так, как раньше'

S = Mv & (np.arange(L) >= WARM)[None]; qs, qt = np.where(S)
yq = np.clip(Yv[qs, qt].astype(np.float64), -2, 2); wq = (VALMIX / hv)[famv[qs]]
PE = np.mean([np.clip(np.load(f'{CKPT_DIR}/Pb_valid300.npy') * 0.3, -2, 2), np.load(f'{CKPT_DIR}/P_gru6573_valid300.npy'),
              np.load(f'{CKPT_DIR}/P_corr3k_valid300.npy')], 0).astype(np.float64)
base_r = PE[qs, qt]
def wcorr(y, p, w):
    sw = w.sum(); my, mp = (w * y).sum() / sw, (w * p).sum() / sw
    return (w * (y - my) * (p - mp)).sum() / np.sqrt((w * (y - my) ** 2).sum() * (w * (p - mp) ** 2).sum())
sc = lambda P, m=slice(None): np.mean([wcorr(yq[m, j], P[m, j], np.abs(yq[m, j]) * wq[m]) for j in (0, 1)])
U = np.array([0.1254, 0.2413, 0.4685, 0.6569]); S_r = (Yv[..., 0] + Yv[..., 1]).astype(np.float64)[qs, qt]
print('ансамбль:', round(sc(base_r), 4), '(ожидаю 0.6610)')

сверка valid300 с прежним: 0.0e+00
ансамбль: 0.661 (ожидаю 0.6610)


In [3]:
free_gb = shutil.disk_usage('/content').free / 1e9
N_TRAIN = min(10607, int((free_gb - 8) * 1000 / 4.48))
print(f'свободно {free_gb:.0f} ГБ → соберу {N_TRAIN} последовательностей train (~{N_TRAIN * 4.48 / 1000:.0f} ГБ)')
rows = np.arange(10607) if N_TRAIN == 10607 else np.sort(np.random.default_rng(0).choice(10607, N_TRAIN, replace=False))
if not os.path.exists(f'{D}/trainall_Y.npy'): build16('train', 'trainall', rows)
os.remove(f'{ROOT}/train.parquet'); sh('df -h /content | tail -1')

свободно 81 ГБ → соберу 10607 последовательностей train (~48 ГБ)
  trainall 0/10607 0 с
  trainall 1000/10607 65 с
  trainall 2000/10607 131 с
  trainall 3000/10607 196 с
  trainall 4000/10607 266 с
  trainall 5000/10607 330 с
  trainall 6000/10607 403 с
  trainall 7000/10607 473 с
  trainall 8000/10607 544 с
  trainall 9000/10607 615 с
  trainall 10000/10607 693 с
overlay         113G   56G   58G  50% /


In [ ]:
class Cat:                                                     # склейка двух float16-наборов, отдаёт float32
    def __init__(s, A, B): s.A, s.B, s.n = A, B, len(A)
    def __len__(s): return s.n + len(s.B)
    def __getitem__(s, idx):
        idx = np.asarray(idx); a, b = idx[idx < s.n], idx[idx >= s.n] - s.n
        return np.concatenate([p for p in ((s.A[a] if len(a) else None), (s.B[b] if len(b) else None)) if p is not None]).astype(np.float32)
Xtr = Cat(np.load(f'{D}/trainall_X16.npy', mmap_mode='r'), np.load(f'{D}/validrest_X16.npy', mmap_mode='r'))
Ytr = np.concatenate([np.load(f'{D}/trainall_Y.npy'), np.load(f'{D}/validrest_Y.npy')])
Xva, Yva, Mva, fam_v, have = Xv, Yv, Mv, famv, hv
print('обучающих последовательностей:', len(Xtr))
torch.manual_seed(0)
net_all = train(Net().to(dev), epochs=5, tag='gru_all')
torch.save(net_all.state_dict(), f'{CKPT_DIR}/gru_all_last.pt')
print('последняя эпоха:', mix_score(net_all), '| ориентир gru6573: 0.498 / 0.657')

обучающих последовательностей: 12180


/content/drive/MyDrive/wnn/lib.py:75: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).to(dev)


эпоха 1/5 | 476 с | loss 0.9735 | valid все строки 0.486 [0.485 0.486] | is_scored 0.661 [0.659 0.662]
эпоха 2/5 | 586 с | loss 0.9455 | valid все строки 0.489 [0.491 0.488] | is_scored 0.662 [0.656 0.667]
эпоха 3/5 | 587 с | loss 0.9339 | valid все строки 0.495 [0.497 0.493] | is_scored 0.668 [0.665 0.671]
эпоха 4/5 | 590 с | loss 0.9207 | valid все строки 0.501 [0.502 0.499] | is_scored 0.670 [0.666 0.675]
эпоха 5/5 | 591 с | loss 0.9085 | valid все строки 0.502 [0.502 0.501] | is_scored 0.670 [0.665 0.675]
последняя эпоха: {'все строки': 0.501, 'is_scored': 0.657} | ориентир gru6573: 0.498 / 0.657


In [ ]:
net_all.eval(); p_path = f'{CKPT_DIR}/P_gruall_valid300.npy'
with torch.no_grad(): Pa = predict(net_all, Xv, np.arange(len(Xv))).astype(np.float32)
np.save(p_path, Pa)
P = {'base': np.clip(np.load(f'{CKPT_DIR}/Pb_valid300.npy') * 0.3, -2, 2), 'gru': np.load(f'{CKPT_DIR}/P_gru6573_valid300.npy'),
     'corr': np.load(f'{CKPT_DIR}/P_corr3k_valid300.npy'), 'all': Pa}
for combo in (['gru'], ['all'], ['gru', 'all'], ['base', 'gru', 'corr'], ['base', 'all', 'corr'], ['base', 'gru', 'corr', 'all']):
    print(f'{" + ".join(combo):22s} {sc(np.mean([P[k] for k in combo], 0)[qs, qt]):.4f}')

gru                    0.6569
all                    0.6570
gru + all              0.6594
base + gru + corr      0.6610
base + all + corr      0.6612
base + gru + corr + all 0.6615


In [ ]:
import numpy as np, torch, os, time, shutil, zipfile, importlib.util
from types import SimpleNamespace
torch.set_num_threads(2)
SRC, OUT, L_, WARM_, NS = '/content/ens3_src', '/content/ens4x', 20000, 99, 20
CKPT_NEW, AX = f'{CKPT_DIR}/gru_all_last.pt', 0.07
if not os.path.exists(f'{SRC}/weights.npz'):
    os.makedirs(SRC, exist_ok=True); zipfile.ZipFile(f'{CKPT_DIR}/ens3_submission.zip').extractall(SRC)

def load_pm(d, name):
    sp = importlib.util.spec_from_file_location(name, f'{d}/solution.py'); m = importlib.util.module_from_spec(sp); sp.loader.exec_module(m)
    return m.PredictionModel()
def run_pm(pm, X, sid):
    out = np.zeros((len(X), 2), np.float32)
    for t in range(len(X)): out[t] = pm.predict(SimpleNamespace(seq_ix=sid, step_in_seq=t, state=X[t], need_prediction=True))
    return out
sd = torch.load(CKPT_NEW, map_location='cpu'); H = int(sd['gru.weight_hh_l0'].shape[1])
net = Net(H); net.load_state_dict(sd); net.eval()
def torch_out(X):
    with torch.no_grad(): return net(torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32)))[0].numpy()

# ---------- статистики: старый ансамбль из 3 моделей и новая модель ----------
W0 = dict(np.load(f'{SRC}/weights.npz')); assert W0['mats'].dtype == np.uint16
K_old = len(W0['mods']); print('моделей в старом ансамбле:', K_old)
pm_old = load_pm(SRC, 'sol_old'); Xs = np.asarray(Xv[:NS], np.float32); t = time.time()
old = np.stack([run_pm(pm_old, Xs[i], i) for i in range(NS)]).astype(np.float64); print(f'старый ансамбль: {time.time() - t:.0f} с')
old_nc = old - W0['const'].astype(np.float64)
Pn = np.concatenate([torch_out(Xs[i:i + 5]) for i in range(0, NS, 5)]).astype(np.float64)
st = lambda a: (a[:, WARM_:].reshape(-1, 2).mean(0), a[:, WARM_:].reshape(-1, 2).std(0))
mu_o, sd_o = st(old_nc); m_old = st(old)[0]; mu_p, sd_p = st(Pn)
U_ = K_old * (old_nc - mu_o) / sd_o + (Pn - mu_p) / sd_p                    # каждая модель с весом 1/(K_old+1)
s = sd_o / st(U_)[1]; A = K_old * s / sd_o; w = s / sd_p; const_new = m_old - A * mu_o - w * mu_p

# ---------- дописываем новую модель как тип 0 ----------
mats, flat, meta, mods = W0['mats'], W0['flat'], W0['meta'], W0['mods']
mo, fo, r = len(mats), len(flat), len(meta); so = int((mods[:, 2] + 2 * mods[:, 3]).max()); mc, fc, rows = [], [], []
def addm(a):
    global mo
    a = np.ascontiguousarray(a, dtype=np.float16); mc.append(a.view(np.uint16).reshape(-1)); rows.append([mo, a.shape[0], a.shape[1]]); mo += a.size
def addv(v):
    global fo
    v = np.ascontiguousarray(v, dtype=np.float32); fc.append(v); rows.append([fo, 1, len(v)]); fo += len(v)
g = lambda k: sd[k].numpy().astype(np.float32)
addm(g('gru.weight_ih_l0').T); addm(g('gru.weight_hh_l0').T); addv(g('gru.bias_ih_l0')); addv(g('gru.bias_hh_l0'))
addm(g('gru.weight_ih_l1').T); addm(g('gru.weight_hh_l1').T); addv(g('gru.bias_ih_l1')); addv(g('gru.bias_hh_l1'))
addm(g('head.weight').T); addv(g('head.bias'))
Wn = dict(W0)
Wn['mats'] = np.concatenate([mats] + mc); Wn['flat'] = np.concatenate([flat] + fc)
Wn['meta'] = np.vstack([meta, np.array(rows, np.int64)]); Wn['mods'] = np.vstack([mods, np.array([[0, r, so, H]], np.int64)])
Wn['coef'] = np.vstack([A * W0['coef'], w[None, :]]).astype(np.float32); Wn['const'] = const_new.astype(np.float32)
pt = torch_out(W0['selftest_x'][None])[0].astype(np.float64)
Wn['selftest_y'] = (A * (W0['selftest_y'].astype(np.float64) - W0['const']) + w * pt + const_new).astype(np.float32)

# ---------- solution.py: тот же движок + перекрёстное смешивание в конце ----------
if os.path.exists(OUT): shutil.rmtree(OUT)
os.makedirs(OUT); np.savez(f'{OUT}/weights.npz', **Wn)
src = open(f'{SRC}/solution.py').read(); assert 'import numpy as np' in src
src += f'''

# ---- перекрёстное смешивание таргетов, A = {AX} ----
_BasePredictionModel = PredictionModel
class PredictionModel(_BasePredictionModel):
    _C, _A = {1 - AX!r}, {AX!r}
    def predict(self, data_point):
        out = super().predict(data_point)
        if out is None: return None
        o0, o1 = float(out[0]), float(out[1])
        return np.array((self._C * o0 - self._A * o1, self._C * o1 - self._A * o0), dtype=np.float32)
'''
open(f'{OUT}/solution.py', 'w').write(src)

# ---------- проверки ----------
pm_new = load_pm(OUT, 'sol_new'); print('самопроверка solution.py пройдена; моделей:', len(Wn['mods']))
M = np.array([[1 - AX, -AX], [-AX, 1 - AX]]); worst = 0.0; Xe = np.asarray(Xv[20:22], np.float32)
for i in range(2):
    o_old = run_pm(pm_old, Xe[i], 100 + i).astype(np.float64); o_new = run_pm(pm_new, Xe[i], 100 + i).astype(np.float64)
    exp = (A * (o_old - W0['const']) + w * torch_out(Xe[i:i + 1])[0] + const_new) @ M
    worst = max(worst, np.abs(o_new - exp).max())
print('макс. расхождение с формулой:', worst, '(нужно < 1e-3)')
d1, d2 = run_pm(pm_new, Xe[0][:3000], 300), run_pm(pm_new, Xe[0][:3000], 301)
print('детерминизм:', np.array_equal(d1, d2), '| конечны:', bool(np.isfinite(d1).all()))
tm = {}
for nm, pm in (('старый', pm_old), ('новый', pm_new)):
    t = time.time(); run_pm(pm, Xe[0], 400); tm[nm] = time.time() - t
print(f'оценка на платформе ≈ {26.1 * tm["новый"] / tm["старый"]:.0f} мин (от 26 мин у ens3)')
shutil.make_archive('/content/ens4x_submission', 'zip', OUT); shutil.copy('/content/ens4x_submission.zip', f'{CKPT_DIR}/')
print('готово: /content/ens4x_submission.zip (копия на Drive)')

моделей в старом ансамбле: 3
старый ансамбль: 41 с
самопроверка solution.py пройдена; моделей: 4
макс. расхождение с формулой: 0.0047243249895184 (нужно < 1e-3)
детерминизм: True | конечны: True
оценка на платформе ≈ 32 мин (от 26 мин у ens3)
готово: /content/ens4x_submission.zip (копия на Drive)


In [ ]:
from scipy.signal import lfilter
yc = np.clip(Yv.astype(np.float64), -2, 2)
f3 = famv[qs] == 3
sc3 = lambda P: np.mean([wcorr(yq[f3, j], P[f3, j], np.abs(yq[f3, j])) for j in (0, 1)])
print(f'ансамбль: все {sc(base_r):.4f} | F3 {sc3(base_r):.4f}')

# ---------- А. оракул сверху: свои a + b·p для каждой последовательности по её же строкам ----------
PA = base_r.copy()
for s in range(300):
    r = np.where(qs == s)[0]
    if len(r) < 20: continue
    for j in (0, 1):
        Z = np.column_stack([np.ones(len(r)), base_r[r, j]]); sw = np.sqrt(np.abs(yq[r, j]))
        b = np.linalg.lstsq(Z * sw[:, None], yq[r, j] * sw, rcond=None)[0]; PA[r, j] = Z @ b
print(f'А. оракул (подгонка по своим же строкам): все {sc(PA):.4f} | F3 {sc3(PA):.4f}')

# ---------- Б. причинная адаптация на ИСТИННЫХ метках с задержкой (тоже оракул: метки нам недоступны) ----------
LAMB, LAGS, TAUS = 0.999, (200, 500), (30.0, 100.0, 300.0)
PB = {(lg, tau): base_r.copy() for lg in LAGS for tau in TAUS}
post = np.arange(L) >= WARM
for s in range(300):
    r = np.where(qs == s)[0]
    if not len(r): continue
    Z = np.column_stack([np.ones(L), PE[s, :, 0], PE[s, :, 1]])
    for j in (0, 1):
        w = np.abs(yc[s, :, j]) * post
        Azz = lfilter([1], [1, -LAMB], (Z[:, :, None] * Z[:, None, :] * w[:, None, None]).reshape(L, 9), axis=0)
        bz = lfilter([1], [1, -LAMB], Z * (w * yc[s, :, j])[:, None], axis=0)
        prior = np.array([0., 1., 0.]) if j == 0 else np.array([0., 0., 1.])   # «оставить прогноз как есть»
        for lg in LAGS:
            tq = qt[r] - lg; ok = tq >= WARM; ti = np.maximum(tq, 0)
            for tau in TAUS:
                beta = np.linalg.solve(Azz[ti].reshape(-1, 3, 3) + tau * np.eye(3), (bz[ti] + tau * prior)[..., None])[..., 0]
                pred = (Z[qt[r]] * beta).sum(1); PB[(lg, tau)][r[ok], j] = pred[ok]
for (lg, tau), P in PB.items():
    print(f'Б. задержка {lg}, τ={tau:<5}: все {sc(P):.4f} ({sc(P) - sc(base_r):+.4f}) | F3 {sc3(P):.4f}')

ансамбль: все 0.6610 | F3 0.4022
А. оракул (подгонка по своим же строкам): все 0.7137 | F3 0.5411
Б. задержка 200, τ=30.0 : все 0.6415 (-0.0195) | F3 0.4731
Б. задержка 200, τ=100.0: все 0.6623 (+0.0012) | F3 0.4930
Б. задержка 200, τ=300.0: все 0.6815 (+0.0205) | F3 0.5081
Б. задержка 500, τ=30.0 : все 0.5792 (-0.0819) | F3 0.3312
Б. задержка 500, τ=100.0: все 0.6051 (-0.0559) | F3 0.3562
Б. задержка 500, τ=300.0: все 0.6331 (-0.0279) | F3 0.3836


In [ ]:
!pip install -q onnx onnxruntime
import onnxruntime as ort
src = '/content/wnn_connectome_starterpack/baseline/solution.py'
assert os.path.exists(src), 'нет папки baseline из стартпака'
net_c = Net(128); net_c.load_state_dict(torch.load(f'{CKPT_DIR}/gru_all_last.pt', map_location='cpu')); net_c.eval()
os.makedirs('/content/sub_all', exist_ok=True); z = torch.zeros(1, 1, 128)
torch.onnx.export(Step(net_c), (torch.zeros(1, 1, 112), z, z.clone()), '/content/sub_all/model.onnx',
                  input_names=['features', 'hidden_0', 'hidden_1'], output_names=['prediction', 'hidden_0_out', 'hidden_1_out'],
                  opset_version=17, dynamo=False)
s_ = open(src).read().replace('baseline.onnx', 'model.onnx')
s_ = s_.replace('return np.asarray(prediction[0, 0], dtype=np.float32)',
                'return np.nan_to_num(np.asarray(prediction[0, 0], dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)')
assert 'nan_to_num' in s_, 'строка return в solution.py другая — пришли её'
open('/content/sub_all/solution.py', 'w').write(s_)
sess = ort.InferenceSession('/content/sub_all/model.onnx', providers=['CPUExecutionProvider'])
x = np.ascontiguousarray(Xv[0, :500], np.float32); h0 = np.zeros((1, 1, 128), np.float32); h1 = h0.copy(); o = []
for t in range(500):
    p, h0, h1 = sess.run(None, {'features': x[t][None, None], 'hidden_0': h0, 'hidden_1': h1}); o.append(p[0, 0])
with torch.no_grad(): ref = net_c(torch.from_numpy(x[None]))[0][0].numpy()
print('расхождение ONNX и PyTorch:', np.abs(np.array(o) - ref).max(), '(нужно < 1e-4)')
shutil.make_archive('/content/gru_all_submission', 'zip', '/content/sub_all'); shutil.copy('/content/gru_all_submission.zip', f'{CKPT_DIR}/')
print('готово: /content/gru_all_submission.zip')

/tmp/ipykernel_1165/173617564.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(Step(net_c), (torch.zeros(1, 1, 112), z, z.clone()), '/content/sub_all/model.onnx',
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4604: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with GRU can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


расхождение ONNX и PyTorch: 1.0728836e-06 (нужно < 1e-4)
готово: /content/gru_all_submission.zip


In [ ]:
# ---------- Г. длина окна: автокорреляция приращений D в ценовых событиях ----------
LG = np.arange(1, 401)
for f in (3, 2):
    num = np.zeros(len(LG)); d1 = np.zeros(len(LG)); d2 = np.zeros(len(LG)); nev = []
    for s in np.where(famv == f)[0]:
        x = np.asarray(Xv[s, :, :22], np.float32)
        ev = np.flatnonzero(np.r_[False, (np.diff(x, axis=0) != 0).any(1)]); ev = ev[ev >= WARM]; nev.append(len(ev))
        dD = np.diff((Yv[s, ev, 0] - Yv[s, ev, 1]).astype(np.float64)); dD -= dD.mean()
        for i, lg in enumerate(LG):
            if lg >= len(dD): break
            a, b = dD[:-lg], dD[lg:]; num[i] += (a * b).sum(); d1[i] += (a * a).sum(); d2[i] += (b * b).sum()
    ac = num / np.sqrt(d1 * d2); o = np.argsort(ac)[:5]
    print(f'Г. F{f}: самые отрицательные лаги (в ценовых событиях): {[(int(LG[i]), round(float(ac[i]), 3)) for i in o]} '
          f'| лаг 1: {ac[0]:.3f} | ценовых событий на последовательность ≈ {np.mean(nev):.0f}')

# ---------- восстановление y(t) по ценам i0 на шагах (t, t+W] ----------
half = np.arange(300) % 2 == 0
def wls(Z, yv, w, lam=1e-3):
    A = Z.T @ (Z * w[:, None]); A[np.diag_indices_from(A)] += lam * np.trace(A) / len(A)
    return np.linalg.solve(A, Z.T @ (w * yv))
print('\nкорреляция восстановленного y(t) с истинным (честное деление пополам), по семействам F0..F3:')
for W in (50, 100, 200, 300, 500):
    ok = qt + W < L; rr = np.where(ok)[0]; Fe = np.zeros((len(rr), 67), np.float32); Fe[:, 0] = 1
    for s in range(300):
        m = qs[rr] == s
        if not m.any(): continue
        x = np.asarray(Xv[s, :, :22], np.float32); d = np.diff(x, axis=0, prepend=x[:1])
        C1 = np.cumsum(np.sign(d), 0); C3 = np.cumsum(d != 0, 0); t = qt[rr][m]
        Fe[m, 1:23] = C1[t + W] - C1[t]; Fe[m, 23:45] = x[t + W] - x[t]; Fe[m, 45:67] = C3[t + W] - C3[t]
    res = []
    for f in range(4):
        cs = []
        for j in (0, 1):
            P = np.zeros(len(rr))
            for a in (True, False):
                tr = (half[qs[rr]] == a) & (famv[qs[rr]] == f); te = (half[qs[rr]] != a) & (famv[qs[rr]] == f)
                yv = yq[rr, j]; P[te] = Fe[te] @ wls(Fe[tr], yv[tr], np.abs(yv[tr]))
            m = famv[qs[rr]] == f; cs.append(wcorr(yq[rr][m, j], P[m], np.abs(yq[rr][m, j])))
        res.append(round(float(np.mean(cs)), 3))
    print(f'  W = {W:>3} шагов: {res}   (для сравнения ансамбль ≈ [0.78, 0.75, 0.72, 0.40] без будущего)')

Г. F3: самые отрицательные лаги (в ценовых событиях): [(3, -0.503), (9, -0.419), (2, -0.412), (33, -0.319), (15, -0.298)] | лаг 1: 0.223 | ценовых событий на последовательность ≈ 6515
Г. F2: самые отрицательные лаги (в ценовых событиях): [(2, -0.339), (1, -0.13), (6, -0.089), (5, -0.084), (10, -0.053)] | лаг 1: -0.130 | ценовых событий на последовательность ≈ 2155

корреляция восстановленного y(t) с истинным (честное деление пополам), по семействам F0..F3:
  W =  50 шагов: [0.792, 0.744, 0.694, 0.367]   (для сравнения ансамбль ≈ [0.78, 0.75, 0.72, 0.40] без будущего)
  W = 100 шагов: [0.805, 0.752, 0.709, 0.373]   (для сравнения ансамбль ≈ [0.78, 0.75, 0.72, 0.40] без будущего)
  W = 200 шагов: [0.756, 0.691, 0.673, 0.315]   (для сравнения ансамбль ≈ [0.78, 0.75, 0.72, 0.40] без будущего)
  W = 300 шагов: [0.698, 0.629, 0.614, 0.255]   (для сравнения ансамбль ≈ [0.78, 0.75, 0.72, 0.40] без будущего)
  W = 500 шагов: [0.581, 0.528, 0.518, 0.18]   (для сравнения ансамбль ≈ [0.78, 0.75, 0

In [ ]:
base = sc(base_r); print(f'ансамбль: все {base:.4f} | F3 {sc3(base_r):.4f}')

# ---------- 1. разложение оракула А и 2. перекрёстная подгонка по блокам времени ----------
Pa_, Pb_, Pab, Pcv = (base_r.copy() for _ in range(4)); blk = (qt // 2000) % 2
for s in range(300):
    r = np.where(qs == s)[0]
    if len(r) < 20: continue
    for j in (0, 1):
        y, p, w = yq[r, j], base_r[r, j], np.abs(yq[r, j])
        Pa_[r, j] = p + (w * (y - p)).sum() / w.sum()                       # только сдвиг a (b = 1)
        Pb_[r, j] = p * (w * y * p).sum() / max((w * p * p).sum(), 1e-12)  # только масштаб b (a = 0)
        Z = np.column_stack([np.ones(len(r)), p])
        fit = lambda m: np.linalg.lstsq(Z[m] * np.sqrt(w[m])[:, None], y[m] * np.sqrt(w[m]), rcond=None)[0]
        Pab[r, j] = Z @ fit(np.ones(len(r), bool))
        for b in (0, 1):                                                    # подгонка на одних блоках, применение к другим
            tr, te = blk[r] != b, blk[r] == b
            if tr.sum() > 10 and te.any(): Pcv[r[te], j] = Z[te] @ fit(tr)
for name, P in (('оракул a + b (как раньше)', Pab), ('только сдвиг a', Pa_), ('только масштаб b', Pb_),
                ('a + b, перекрёстно по блокам 2000 шагов', Pcv)):
    print(f'  {name:40s} все {sc(P):.4f} ({sc(P) - base:+.4f}) | F3 {sc3(P):.4f}')

# ---------- 3. причинная без забывания, задержка 500 ----------
post = np.arange(L) >= WARM; TAUS = (100.0, 300.0, 1000.0); P3 = {tau: base_r.copy() for tau in TAUS}
for s in range(300):
    r = np.where(qs == s)[0]
    if not len(r): continue
    Z = np.column_stack([np.ones(L), PE[s, :, 0], PE[s, :, 1]])
    for j in (0, 1):
        w = np.abs(yc[s, :, j]) * post
        Azz = np.cumsum((Z[:, :, None] * Z[:, None, :] * w[:, None, None]).reshape(L, 9), 0)
        bz = np.cumsum(Z * (w * yc[s, :, j])[:, None], 0); prior = np.array([0., 1., 0.]) if j == 0 else np.array([0., 0., 1.])
        tq = qt[r] - 500; ok = tq >= WARM; ti = np.maximum(tq, 0)
        for tau in TAUS:
            beta = np.linalg.solve(Azz[ti].reshape(-1, 3, 3) + tau * np.eye(3), (bz[ti] + tau * prior)[..., None])[..., 0]
            P3[tau][r[ok], j] = (Z[qt[r]] * beta).sum(1)[ok]
for tau, P in P3.items(): print(f'  без забывания, задержка 500, τ={tau:<6} все {sc(P):.4f} ({sc(P) - base:+.4f}) | F3 {sc3(P):.4f}')

# ---------- 4. проверка Г по каждой последовательности F3 отдельно ----------
LGs = np.arange(1, 61); best = []
for s in np.where(famv == 3)[0]:
    x = np.asarray(Xv[s, :, :22], np.float32)
    ev = np.flatnonzero(np.r_[False, (np.diff(x, axis=0) != 0).any(1)]); ev = ev[ev >= WARM]
    dD = np.diff((Yv[s, ev, 0] - Yv[s, ev, 1]).astype(np.float64)); dD -= dD.mean(); v = (dD * dD).sum()
    ac = np.array([(dD[:-lg] * dD[lg:]).sum() / v for lg in LGs])
    o = np.argsort(ac)[:2]; best.append((LGs[o[0]], ac[o[0]], LGs[o[1]], ac[o[1]]))
best = np.array(best)
v, c = np.unique(best[:, 0].astype(int), return_counts=True); o = np.argsort(-c)
print('\nГ по последовательностям F3: самый отрицательный лаг (лаг: число последовательностей):', [(int(v[i]), int(c[i])) for i in o[:8]])
print('  значение в этом пике, квантили 10/50/90%:', np.round(np.quantile(best[:, 1], [0.1, 0.5, 0.9]), 3),
      '| доля последовательностей с пиком ≤ −0.4:', round(float((best[:, 1] <= -0.4).mean()), 2))
print('  второй по силе пик, медиана значения:', round(float(np.median(best[:, 3])), 3))

ансамбль: все 0.6610 | F3 0.4022
  оракул a + b (как раньше)                все 0.7137 (+0.0527) | F3 0.5411
  только сдвиг a                           все 0.6990 (+0.0379) | F3 0.5078
  только масштаб b                         все 0.6790 (+0.0179) | F3 0.4625
  a + b, перекрёстно по блокам 2000 шагов  все 0.6094 (-0.0517) | F3 0.3005
  без забывания, задержка 500, τ=100.0  все 0.6136 (-0.0475) | F3 0.3238
  без забывания, задержка 500, τ=300.0  все 0.6244 (-0.0367) | F3 0.3391
  без забывания, задержка 500, τ=1000.0 все 0.6377 (-0.0234) | F3 0.3586

Г по последовательностям F3: самый отрицательный лаг (лаг: число последовательностей): [(3, 70), (2, 25), (4, 2), (1, 1)]
  значение в этом пике, квантили 10/50/90%: [-0.802 -0.543 -0.347] | доля последовательностей с пиком ≤ −0.4: 0.76
  второй по силе пик, медиана значения: -0.44


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
OFF = (-1, 0, 1, 2, 3, 4)                                       # смещения в ценовых событиях относительно k
Fs, Ts, Ss = [], [], []
for s in np.where(famv == 3)[0]:
    x = np.asarray(Xv[s, :, :22], np.float32); d = np.diff(x, axis=0, prepend=x[:1])
    ev = np.flatnonzero(np.r_[False, (np.diff(x, axis=0) != 0).any(1)]); ev = ev[ev >= WARM]
    y = Yv[s, ev].astype(np.float64); clip = (np.abs(y) >= 2 - 1e-6).any(1)
    D = y[:, 0] - y[:, 1]; k = np.arange(2, len(ev) - 5)
    ok = ~clip[k] & ~clip[k - 1]; k = k[ok]                     # без строк, где таргет обрезан на ±2
    Fs.append(np.concatenate([d[ev[k + o]] for o in OFF], 1)); Ts.append(D[k] - D[k - 1]); Ss.append(np.full(len(k), s))
F_ = np.concatenate(Fs).astype(np.float32); T_ = np.concatenate(Ts); S_ = np.concatenate(Ss)
print('событий F3 для проверки:', len(T_), '| доля нулевых изменений D:', round(float((np.abs(T_) < 1e-9).mean()), 3))

even = np.isin(S_, np.where(famv == 3)[0][::2]); rng = np.random.default_rng(0)
for name in ('линейная', 'бустинг'):
    P = np.zeros(len(T_))
    for a in (True, False):
        tr, te = np.where(even == a)[0], np.where(even != a)[0]
        if name == 'линейная':
            Z = np.column_stack([np.ones(len(F_)), F_]); A = Z[tr].T @ Z[tr]; A[np.diag_indices_from(A)] += 1e-3 * np.trace(A) / len(A)
            P[te] = Z[te] @ np.linalg.solve(A, Z[tr].T @ T_[tr])
        else:
            sub = rng.choice(tr, min(200_000, len(tr)), replace=False)
            m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, max_leaf_nodes=63, early_stopping=False, random_state=0)
            P[te] = m.fit(F_[sub], T_[sub]).predict(F_[te])
    print(f'{name:9s}: corr(предсказанное, истинное изменение D) = {np.corrcoef(P, T_)[0, 1]:.3f} | '
          f'доля объяснённой дисперсии R² = {1 - ((T_ - P) ** 2).mean() / T_.var():.3f}', flush=True)

событий F3 для проверки: 631102 | доля нулевых изменений D: 0.35
линейная : corr(предсказанное, истинное изменение D) = 0.924 | доля объяснённой дисперсии R² = 0.854
бустинг  : corr(предсказанное, истинное изменение D) = 0.940 | доля объяснённой дисперсии R² = 0.884


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
base = sc(base_r); print(f'ансамбль: все {base:.4f} | F3 {sc3(base_r):.4f}')
half = np.arange(300) % 2 == 0

# ---------- 1а. потолок: глобальный линейный стек [прогноз, истинный y(t − d)] ----------
def wls(Z, yv, w, lam=1e-4):
    A = Z.T @ (Z * w[:, None]); A[np.diag_indices_from(A)] += lam * np.trace(A) / len(A)
    return np.linalg.solve(A, Z.T @ (w * yv))
def stack(extra):
    P = np.zeros_like(base_r); Z = np.column_stack([np.ones(len(qs)), base_r] + ([extra] if extra is not None else []))
    for a in (True, False):
        tr, te = half[qs] == a, half[qs] != a
        for j in (0, 1): P[te, j] = Z[te] @ wls(Z[tr], yq[tr, j], np.abs(yq[tr, j]) * wq[tr])
    return P
P0 = stack(None); print(f'1а. контроль (стек без меток): все {sc(P0):.4f}')
for lg in (10, 20, 50, 100, 200):
    P = stack(yc[qs, qt - lg]); print(f'1а. + истинный y(t−{lg:<3}): все {sc(P):.4f} ({sc(P) - sc(P0):+.4f} к контролю) | F3 {sc3(P):.4f}')

# ---------- 1б. потолок: подстройка как в Б (забывание 0.999, τ = 300) при коротких задержках ----------
from scipy.signal import lfilter
post = np.arange(L) >= WARM; LGS = (20, 50, 100); PB = {lg: base_r.copy() for lg in LGS}
for s in range(300):
    r = np.where(qs == s)[0]
    if not len(r): continue
    Z = np.column_stack([np.ones(L), PE[s, :, 0], PE[s, :, 1]])
    for j in (0, 1):
        w = np.abs(yc[s, :, j]) * post
        Azz = lfilter([1], [1, -0.999], (Z[:, :, None] * Z[:, None, :] * w[:, None, None]).reshape(L, 9), axis=0)
        bz = lfilter([1], [1, -0.999], Z * (w * yc[s, :, j])[:, None], axis=0); prior = np.eye(3)[1 + j]
        for lg in LGS:
            tq = qt[r] - lg; ok = tq >= WARM; ti = np.maximum(tq, 0)
            beta = np.linalg.solve(Azz[ti].reshape(-1, 3, 3) + 300 * np.eye(3), (bz[ti] + 300 * prior)[..., None])[..., 0]
            PB[lg][r[ok], j] = (Z[qt[r]] * beta).sum(1)[ok]
for lg, P in PB.items(): print(f'1б. подстройка, задержка {lg:<3}: все {sc(P):.4f} ({sc(P) - base:+.4f}) | F3 {sc3(P):.4f}')

# ---------- 2. раскодируется ли УРОВЕНЬ D и S на событии k по событиям k..k+4 ----------
OFF = (0, 1, 2, 3, 4); rng = np.random.default_rng(0)
for f in (3, 2):
    Fs, Ds, Ss, Sq = [], [], [], []
    for s in np.where(famv == f)[0]:
        x = np.asarray(Xv[s, :, :22], np.float32); d = np.diff(x, axis=0, prepend=x[:1])
        ev = np.flatnonzero(np.r_[False, (np.diff(x, axis=0) != 0).any(1)]); ev = ev[ev >= WARM]
        y = Yv[s, ev].astype(np.float64); k = np.arange(1, len(ev) - 5); k = k[~(np.abs(y[k]) >= 2 - 1e-6).any(1)]
        Fs.append(np.concatenate([d[ev[k + o]] for o in OFF] + [x[ev[k]], x[ev[k + 3]]], 1))
        Ds.append(y[k, 0] - y[k, 1]); Ss.append(y[k, 0] + y[k, 1]); Sq.append(np.full(len(k), s))
    F_ = np.concatenate(Fs).astype(np.float32); Sq = np.concatenate(Sq); ev_ = np.isin(Sq, np.where(famv == f)[0][::2])
    for name, T_ in (('D', np.concatenate(Ds)), ('S', np.concatenate(Ss))):
        P = np.zeros(len(T_))
        for a in (True, False):
            tr, te = np.where(ev_ == a)[0], np.where(ev_ != a)[0]; sub = rng.choice(tr, min(200_000, len(tr)), replace=False)
            m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, max_leaf_nodes=63, early_stopping=False, random_state=0)
            P[te] = m.fit(F_[sub], T_[sub]).predict(F_[te])
        print(f'2. F{f}: уровень {name} по событиям k..k+4 — corr {np.corrcoef(P, T_)[0, 1]:.3f}', flush=True)

ансамбль: все 0.6610 | F3 0.4022
1а. контроль (стек без меток): все 0.6613
1а. + истинный y(t−10 ): все 0.9450 (+0.2838 к контролю) | F3 0.9479
1а. + истинный y(t−20 ): все 0.9108 (+0.2495 к контролю) | F3 0.9108
1а. + истинный y(t−50 ): все 0.8272 (+0.1660 к контролю) | F3 0.8037
1а. + истинный y(t−100): все 0.7512 (+0.0900 к контролю) | F3 0.6739
1а. + истинный y(t−200): все 0.7005 (+0.0393 к контролю) | F3 0.5487
1б. подстройка, задержка 20 : все 0.7697 (+0.1087) | F3 0.6768
1б. подстройка, задержка 50 : все 0.7446 (+0.0835) | F3 0.6348
1б. подстройка, задержка 100: все 0.7160 (+0.0549) | F3 0.5812
2. F3: уровень D по событиям k..k+4 — corr 0.873
2. F3: уровень S по событиям k..k+4 — corr 0.971
2. F2: уровень D по событиям k..k+4 — corr 0.697
2. F2: уровень S по событиям k..k+4 — corr 0.732


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
XT = np.load('/content/data/trainall_X16.npy', mmap_mode='r'); YT = np.load('/content/data/trainall_Y.npy', mmap_mode='r')
famT = np.concatenate([fam_of(np.asarray(YT[i:i + 500]), verbose=False) for i in range(0, len(YT), 500)])
OFF = (0, 1, 2, 3, 4); rng = np.random.default_rng(0)
def ev_of(x):
    ev = np.flatnonzero(np.r_[False, (np.diff(x[:, :22], axis=0) != 0).any(1)]); return ev[ev >= WARM]
def feats(x, ev):
    d = np.diff(x[:, :22], axis=0, prepend=x[:1, :22]); k = np.arange(0, len(ev) - 4)
    return k, np.concatenate([d[ev[k + o]] for o in OFF] + [x[ev[k], :22], x[ev[k + 3], :22]], 1).astype(np.float32)

# ---------- 1. декодеры D и S по семействам, обучение на train ----------
dec = {}; t0 = time.time()
for f in range(4):
    Fs, Ds, Ss = [], [], []
    for s in np.where(famT == f)[0][:150]:
        x = np.asarray(XT[s], np.float32); y = np.asarray(YT[s], np.float64); ev = ev_of(x); k, F = feats(x, ev)
        yy = y[ev[k]]; ok = ~(np.abs(yy) >= 2 - 1e-6).any(1)
        Fs.append(F[ok]); Ds.append(yy[ok, 0] - yy[ok, 1]); Ss.append(yy[ok, 0] + yy[ok, 1])
    F_ = np.concatenate(Fs); sub = rng.choice(len(F_), min(250_000, len(F_)), replace=False); Zs = np.column_stack([np.ones(len(sub)), F_[sub]])
    for nm, T_ in (('D', np.concatenate(Ds)), ('S', np.concatenate(Ss))):
        gb = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, max_leaf_nodes=63, early_stopping=False, random_state=0).fit(F_[sub], T_[sub])
        A = Zs.T @ Zs; A[np.diag_indices_from(A)] += 1e-3 * np.trace(A) / len(A); lin = np.linalg.solve(A, Zs.T @ T_[sub])
        dec[(f, nm)] = (gb, lin)
    print(f'декодеры F{f}: {len(F_)} событий, {time.time() - t0:.0f} с', flush=True)

# ---------- 2. причинное раскодирование на valid300 ----------
Yd = {m: np.zeros((300, L, 2)) for m in ('true', 'lin', 'gb')}; LAGd = np.full((300, L), 1e4); qual = {m: [] for m in ('lin', 'gb')}
for s in range(300):
    x = np.asarray(Xv[s], np.float32); ev = ev_of(x); k, F = feats(x, ev); f = famv[s]; Z = np.column_stack([np.ones(len(F)), F])
    tru = np.clip(Yv[s, ev[k]].astype(np.float64), -2, 2); out = {'true': tru}
    for m in ('lin', 'gb'):
        Dh = dec[(f, 'D')][0].predict(F) if m == 'gb' else Z @ dec[(f, 'D')][1]
        Sh = dec[(f, 'S')][0].predict(F) if m == 'gb' else Z @ dec[(f, 'S')][1]
        out[m] = np.clip(np.column_stack([(Sh + Dh) / 2, (Sh - Dh) / 2]), -2, 2); qual[m].append((f, out[m], tru))
    avail = ev[k + 4]; t = np.arange(L); ks = np.searchsorted(avail, t, side='right') - 1; has = ks >= 0
    for m in out: Yd[m][s, has] = out[m][ks[has]]
    LAGd[s, has] = t[has] - ev[k[ks[has]]]
print('\nкачество раскодирования на valid300 (corr с истинным таргетом на событиях, среднее по t0/t1), F0..F3:')
for m in ('lin', 'gb'):
    r = []
    for f in range(4):
        P_ = np.concatenate([o for g, o, _ in qual[m] if g == f]); T_ = np.concatenate([tr for g, _, tr in qual[m] if g == f])
        r.append(round(float(np.mean([np.corrcoef(P_[:, j], T_[:, j])[0, 1] for j in (0, 1)])), 3))
    print(f'  {m}: {r}')
print('медианная задержка доступного таргета (шагов), F0..F3:', [int(np.median(LAGd[qs[famv[qs] == f], qt[famv[qs] == f]])) for f in range(4)])

# ---------- 3. честный стек на valid300 ----------
print(f'\nконтроль (стек без меток): все {sc(P0):.4f} | F3 {sc3(P0):.4f}')
for m in ('true', 'lin', 'gb'):
    E = Yd[m][qs, qt]
    P1 = stack(E)
    P2 = stack(np.column_stack([E * (famv[qs] == f)[:, None] for f in range(4)]))
    tag = 'идеальный декодер' if m == 'true' else f'декодер {m}'
    print(f'  {tag:18s} общий вклад: все {sc(P1):.4f} ({sc(P1) - sc(P0):+.4f}) | F3 {sc3(P1):.4f} || '
          f'по семействам: все {sc(P2):.4f} ({sc(P2) - sc(P0):+.4f}) | F3 {sc3(P2):.4f}')

декодеры F0: 1142604 событий, 97 с
декодеры F1: 869490 событий, 182 с
декодеры F2: 368917 событий, 255 с
декодеры F3: 589753 событий, 323 с

качество раскодирования на valid300 (corr с истинным таргетом на событиях, среднее по t0/t1), F0..F3:
  lin: [0.6, 0.648, 0.676, 0.845]
  gb: [0.625, 0.66, 0.685, 0.863]
медианная задержка доступного таргета (шагов), F0..F3: [10, 27, 67, 97]

контроль (стек без меток): все 0.6613 | F3 0.4007
  идеальный декодер  общий вклад: все 0.7640 (+0.1027) | F3 0.5560 || по семействам: все 0.7658 (+0.1046) | F3 0.5558
  декодер lin        общий вклад: все 0.6588 (-0.0025) | F3 0.3951 || по семействам: все 0.6587 (-0.0026) | F3 0.3921
  декодер gb         общий вклад: все 0.6579 (-0.0034) | F3 0.3915 || по семействам: все 0.6576 (-0.0036) | F3 0.3882


In [ ]:
res = []
for s in np.where(famv == 3)[0]:
    x = np.asarray(Xv[s], np.float32); ev = ev_of(x); k, F = feats(x, ev)
    tru = np.clip(Yv[s, ev[k]].astype(np.float64), -2, 2); pe = PE[s, ev[k]]
    Dh, Sh = dec[(3, 'D')][0].predict(F), dec[(3, 'S')][0].predict(F)
    gb = np.clip(np.column_stack([(Sh + Dh) / 2, (Sh - Dh) / 2]), -2, 2)
    res.append((tru - pe, gb - pe))
R1 = np.concatenate([a for a, _ in res]); R2 = np.concatenate([b for _, b in res])
print('F3: corr(раскодированный остаток, истинный остаток) по t0/t1:', [round(float(np.corrcoef(R1[:, j], R2[:, j])[0, 1]), 3) for j in (0, 1)])

F3: corr(раскодированный остаток, истинный остаток) по t0/t1: [0.711, 0.713]


In [ ]:
a_t, a_d, b_t = [], [], []
for s in np.where(famv == 3)[0]:
    x = np.asarray(Xv[s], np.float32); ev = ev_of(x); k, F = feats(x, ev)
    tru = np.clip(Yv[s, ev[k]].astype(np.float64), -2, 2); pe = PE[s, ev[k]]
    Dh, Sh = dec[(3, 'D')][0].predict(F), dec[(3, 'S')][0].predict(F)
    gb = np.clip(np.column_stack([(Sh + Dh) / 2, (Sh - Dh) / 2]), -2, 2)
    j = np.searchsorted(ev[k], ev[k] + 100); ok = j < len(k)                # событие примерно через 100 шагов
    a_t.append((tru - pe)[ok]); a_d.append((gb - pe)[ok]); b_t.append((tru - pe)[j[ok]])
A_t, A_d, B_t = (np.concatenate(v) for v in (a_t, a_d, b_t))
for j in (0, 1):
    print(f't{j}: corr(истинный остаток сейчас, истинный через 100 шагов) = {np.corrcoef(A_t[:, j], B_t[:, j])[0, 1]:.3f} | '
          f'corr(раскодированный остаток сейчас, истинный через 100 шагов) = {np.corrcoef(A_d[:, j], B_t[:, j])[0, 1]:.3f}')

t0: corr(истинный остаток сейчас, истинный через 100 шагов) = 0.106 | corr(раскодированный остаток сейчас, истинный через 100 шагов) = 0.014
t1: corr(истинный остаток сейчас, истинный через 100 шагов) = 0.088 | corr(раскодированный остаток сейчас, истинный через 100 шагов) = 0.009


In [ ]:
f3s = np.where(famv == 3)[0]

# ---------- 1. правило часов: при изменении каких колонок цены меняется таргет ----------
CH, TC = [], []
for s in f3s:
    x = np.asarray(Xv[s, :, :22], np.float32); y = Yv[s]
    CH.append((np.diff(x, axis=0) != 0)[WARM:]); TC.append((np.diff(y, axis=0) != 0).any(1)[WARM:])
CH, TC = np.concatenate(CH), np.concatenate(TC)
both = (CH & TC[:, None]).sum(0); pc = both / np.maximum(CH.sum(0), 1); pt = both / TC.sum()
o = np.argsort(-(2 * pc * pt / (pc + pt + 1e-12)))
print('F3, колонки цены i0: (колонка, P(смена таргета | колонка изменилась), P(колонка изменилась | смена таргета))')
print('  ', [(int(c), round(float(pc[c]), 3), round(float(pt[c]), 3)) for c in o[:10]])
sel, best_f1 = [], 0
for _ in range(6):
    cand = []
    for c in range(22):
        if c in sel: continue
        u = CH[:, sel + [c]].any(1); tp = (u & TC).sum(); p_, r_ = tp / max(u.sum(), 1), tp / TC.sum()
        cand.append((2 * p_ * r_ / (p_ + r_ + 1e-12), c, p_, r_))
    f1, c, p_, r_ = max(cand)
    if f1 <= best_f1 + 1e-4: break
    sel.append(c); best_f1 = f1; print(f'  набор {sel}: точность {p_:.3f}, полнота {r_:.3f}, F1 {f1:.3f}')
print('  (для сравнения «любая из 22 цен»: точность', round(float(((CH.any(1)) & TC).sum() / CH.any(1).sum()), 3), ', полнота 1.000)')

# ---------- 2. автокорреляция таргета t0 в трёх часах ----------
def acs(series_list, lags):
    out = []
    for lg in lags:
        a = np.concatenate([v[:-lg] for v in series_list if len(v) > lg]); b = np.concatenate([v[lg:] for v in series_list if len(v) > lg])
        out.append(round(float(np.corrcoef(a, b)[0, 1]), 3))
    return out
by_step, by_ev, by_tc = [], [], []
for s in f3s:
    y0 = np.clip(Yv[s, WARM:, 0], -2, 2).astype(np.float64); by_step.append(y0)
    ev = ev_of(np.asarray(Xv[s], np.float32)); by_ev.append(np.clip(Yv[s, ev, 0], -2, 2).astype(np.float64))
    ch = np.flatnonzero(np.r_[True, np.diff(Yv[s, WARM:, 0]) != 0]); by_tc.append(y0[ch])
print('\nавтокорреляция t0 в F3:')
print('  по шагам,          лаги 1/5/10/50/100:', acs(by_step, (1, 5, 10, 50, 100)))
print('  по ценовым событиям, лаги 1/2/3/5/10/30:', acs(by_ev, (1, 2, 3, 5, 10, 30)))
print('  по сменам таргета,  лаги 1/2/3/5/10/30:', acs(by_tc, (1, 2, 3, 5, 10, 30)))

# ---------- длительность «нырков» F3 (t0 ≤ −3u) в шагах и в сменах таргета ----------
dur_s, dur_c = [], []
for s in f3s:
    y = Yv[s, WARM:, 0]; e = code3(np.clip(y, -2, 2)) <= -3
    st = np.flatnonzero(e & ~np.r_[False, e[:-1]]); en = np.flatnonzero(e & ~np.r_[e[1:], False])
    for a, b in zip(st, en): dur_s.append(b - a + 1); dur_c.append(int((np.diff(y[a:b + 1]) != 0).sum()))
print('\nнырки F3: длительность в шагах (квантили 25/50/75/90%):', np.quantile(dur_s, [0.25, 0.5, 0.75, 0.9]).astype(int),
      '| смен таргета внутри (те же квантили):', np.quantile(dur_c, [0.25, 0.5, 0.75, 0.9]).astype(int))

# ---------- 3. идеальный декодер: выигрыш в зависимости от лага в ценовых событиях ----------
print(f'\nидеальный декодер, контроль {sc(P0):.4f}:')
for Le in (4, 6, 8, 12, 20):
    E = np.zeros((300, L, 2))
    for s in range(300):
        ev = ev_of(np.asarray(Xv[s], np.float32)); k = np.arange(len(ev) - Le)
        avail = ev[k + Le]; t = np.arange(L); ks = np.searchsorted(avail, t, side='right') - 1; has = ks >= 0
        E[s, has] = np.clip(Yv[s, ev[k[ks[has]]]], -2, 2)
    P = stack(E[qs, qt]); print(f'  лаг {Le:>2} событий: все {sc(P):.4f} ({sc(P) - sc(P0):+.4f}) | F3 {sc3(P):.4f}', flush=True)

F3, колонки цены i0: (колонка, P(смена таргета | колонка изменилась), P(колонка изменилась | смена таргета))
   [(12, 0.651, 1.0), (13, 0.651, 1.0), (14, 0.651, 1.0), (15, 0.651, 1.0), (11, 0.651, 1.0), (18, 0.651, 1.0), (17, 0.651, 1.0), (16, 0.651, 1.0), (19, 0.651, 1.0), (20, 0.651, 1.0)]
  набор [21]: точность 0.651, полнота 1.000, F1 0.789
  (для сравнения «любая из 22 цен»: точность 0.651 , полнота 1.000)

автокорреляция t0 в F3:
  по шагам,          лаги 1/5/10/50/100: [0.951, 0.845, 0.871, 0.678, 0.511]
  по ценовым событиям, лаги 1/2/3/5/10/30: [0.489, 0.043, 0.045, 0.276, -0.059, 0.293]
  по сменам таргета,  лаги 1/2/3/5/10/30: [-0.211, 0.737, -0.365, -0.408, 0.552, 0.468]

нырки F3: длительность в шагах (квантили 25/50/75/90%): [  4  14  76 208] | смен таргета внутри (те же квантили): [0 0 0 2]

идеальный декодер, контроль 0.6613:
  лаг  4 событий: все 0.7640 (+0.1027) | F3 0.5560
  лаг  6 событий: все 0.7038 (+0.0426) | F3 0.4246
  лаг  8 событий: все 0.6979 (+0.0366) | F3 

In [ ]:
D = '/content/data'; assert L == 20000 and WARM == 99, 'затёрты L или WARM — перезапусти %run lib.py'
def evfeat(x):                                                  # x: (L, 112) → (L, 8), всё причинно
    n = len(x); d = np.diff(x[:, :22], axis=0, prepend=x[:1, :22]); ch = (d != 0).any(1); ch[0] = False
    m = np.where(ch, d.mean(1), 0.0); ev = np.flatnonzero(ch); cnt = np.cumsum(ch)
    cs = np.concatenate([[0.0], np.cumsum(m[ev])]); out = np.zeros((n, 8), np.float32); t = np.arange(n)
    for i, k in enumerate((1, 2, 3, 5, 10)): out[:, i] = cs[cnt] - cs[np.maximum(cnt - k, 0)]    # сумма сдвигов за k событий
    last = ev[np.maximum(cnt - 1, 0)] if len(ev) else np.zeros(n, int)
    out[:, 5] = np.log1p(np.where(cnt > 0, t - last, n))                                          # шагов с последнего события
    c0 = np.concatenate([[0], cnt])
    for i, w in ((6, 50), (7, 200)): out[:, i] = c0[t + 1] - c0[np.maximum(t + 1 - w, 0)]          # событий за w шагов
    return out

XA = np.load(f'{D}/trainall_X16.npy', mmap_mode='r')
S_ = np.concatenate([evfeat(np.asarray(XA[i], np.float32)) for i in range(0, 2000, 100)])
MU, SD = S_.mean(0), S_.std(0) + 1e-6; print('масштаб признаков готов')

class WithEv:                                                   # добавляет 8 признаков к 112 на лету
    def __init__(s, X): s.X = X
    def __len__(s): return len(s.X)
    def __getitem__(s, idx):
        if isinstance(idx, slice): idx = np.arange(len(s.X))[idx]
        idx = np.atleast_1d(np.asarray(idx)); b = np.asarray(s.X[idx], np.float32)
        f = (np.stack([evfeat(v) for v in b]) - MU) / SD
        return np.concatenate([b, f.astype(np.float32)], -1)

class Cat:
    def __init__(s, A, B): s.A, s.B, s.n = A, B, len(A)
    def __len__(s): return s.n + len(s.B)
    def __getitem__(s, idx):
        idx = np.asarray(idx); a, b = idx[idx < s.n], idx[idx >= s.n] - s.n
        return np.concatenate([p for p in ((s.A[a] if len(a) else None), (s.B[b] if len(b) else None)) if p is not None]).astype(np.float32)

Xtr = WithEv(Cat(XA, np.load(f'{D}/validrest_X16.npy', mmap_mode='r')))
Ytr = np.concatenate([np.load(f'{D}/trainall_Y.npy'), np.load(f'{D}/validrest_Y.npy')])
Xva, Yva, Mva, fam_v, have = WithEv(Xv), Yv, Mv, famv, hv
print('обучающих:', len(Xtr), '| вход:', Xtr[np.array([0])].shape[-1])
torch.manual_seed(0)
net_ev = train(Net2(120).to(dev), epochs=5, tag='gru_ev')
torch.save(net_ev.state_dict(), f'{CKPT_DIR}/gru_ev_last.pt')

Pe = predict(net_ev, Xva, np.arange(300)).astype(np.float32); np.save(f'{CKPT_DIR}/P_gruev_valid300.npy', Pe)
Pa = np.load(f'{CKPT_DIR}/P_gruall_valid300.npy')
print(f'\nс событийными признаками: все {sc(Pe[qs, qt]):.4f} | F3 {sc3(Pe[qs, qt]):.4f}')
print(f'контроль gru_all (те же данные и сид): все {sc(Pa[qs, qt]):.4f} | F3 {sc3(Pa[qs, qt]):.4f}')

масштаб признаков готов
обучающих: 12180 | вход: 120
эпоха 1/5 | 726 с | loss 0.9745 | valid все строки 0.483 [0.483 0.483] | is_scored 0.658 [0.657 0.659]
эпоха 2/5 | 720 с | loss 0.9474 | valid все строки 0.488 [0.49  0.487] | is_scored 0.665 [0.66  0.671]
эпоха 3/5 | 717 с | loss 0.9355 | valid все строки 0.491 [0.492 0.49 ] | is_scored 0.666 [0.662 0.67 ]
эпоха 4/5 | 720 с | loss 0.9226 | valid все строки 0.500 [0.5   0.501] | is_scored 0.672 [0.667 0.677]
эпоха 5/5 | 714 с | loss 0.9099 | valid все строки 0.501 [0.5   0.501] | is_scored 0.672 [0.668 0.677]

с событийными признаками: все 0.6600 | F3 0.4031
контроль gru_all (те же данные и сид): все 0.6570 | F3 0.3889


In [ ]:
P = {'base': np.clip(np.load(f'{CKPT_DIR}/Pb_valid300.npy') * 0.3, -2, 2), 'gru': np.load(f'{CKPT_DIR}/P_gru6573_valid300.npy'),
     'corr': np.load(f'{CKPT_DIR}/P_corr3k_valid300.npy'), 'all': np.load(f'{CKPT_DIR}/P_gruall_valid300.npy'),
     'ev': np.load(f'{CKPT_DIR}/P_gruev_valid300.npy')}
for combo in (['all'], ['ev'], ['all', 'ev'], ['gru', 'all'], ['gru', 'ev'], ['base', 'gru', 'corr'],
              ['base', 'gru', 'corr', 'all'], ['base', 'gru', 'corr', 'ev'], ['base', 'gru', 'corr', 'all', 'ev']):
    E = np.mean([P[k] for k in combo], 0)[qs, qt]; print(f'{" + ".join(combo):26s} все {sc(E):.4f} | F3 {sc3(E):.4f}')

all                        все 0.6570 | F3 0.3889
ev                         все 0.6600 | F3 0.4031
all + ev                   все 0.6611 | F3 0.3993
gru + all                  все 0.6594 | F3 0.3944
gru + ev                   все 0.6611 | F3 0.4018
base + gru + corr          все 0.6610 | F3 0.4022
base + gru + corr + all    все 0.6615 | F3 0.4007
base + gru + corr + ev     все 0.6623 | F3 0.4046
base + gru + corr + all + ev все 0.6623 | F3 0.4028


In [ ]:
torch.manual_seed(1)
net_ev1 = train(Net2(120).to(dev), epochs=5, tag='gru_ev_s1')
torch.save(net_ev1.state_dict(), f'{CKPT_DIR}/gru_ev_s1_last.pt')
P['ev1'] = predict(net_ev1, Xva, np.arange(300)).astype(np.float32); np.save(f'{CKPT_DIR}/P_gruev1_valid300.npy', P['ev1'])
for combo in (['ev'], ['ev1'], ['ev', 'ev1'], ['all', 'gru'], ['base', 'gru', 'corr', 'all'], ['base', 'gru', 'corr', 'ev'],
              ['base', 'gru', 'corr', 'ev1'], ['base', 'gru', 'corr', 'ev', 'ev1']):
    E = np.mean([P[k] for k in combo], 0)[qs, qt]; print(f'{" + ".join(combo):26s} все {sc(E):.4f} | F3 {sc3(E):.4f}')

эпоха 1/5 | 706 с | loss 0.9745 | valid все строки 0.483 [0.484 0.482] | is_scored 0.655 [0.655 0.656]
эпоха 2/5 | 719 с | loss 0.9480 | valid все строки 0.486 [0.489 0.484] | is_scored 0.664 [0.66  0.668]
эпоха 3/5 | 726 с | loss 0.9358 | valid все строки 0.488 [0.491 0.486] | is_scored 0.666 [0.663 0.669]
эпоха 4/5 | 695 с | loss 0.9231 | valid все строки 0.499 [0.5   0.497] | is_scored 0.669 [0.665 0.673]
эпоха 5/5 | 691 с | loss 0.9105 | valid все строки 0.500 [0.5   0.499] | is_scored 0.670 [0.667 0.674]
ev                         все 0.6600 | F3 0.4031
ev1                        все 0.6577 | F3 0.3980
ev + ev1                   все 0.6613 | F3 0.4040
all + gru                  все 0.6594 | F3 0.3944
base + gru + corr + all    все 0.6615 | F3 0.4007
base + gru + corr + ev     все 0.6623 | F3 0.4046
base + gru + corr + ev1    все 0.6617 | F3 0.4032
base + gru + corr + ev + ev1 все 0.6625 | F3 0.4048


In [ ]:
np.savez(f'{CKPT_DIR}/evfeat_scale.npz', MU=MU, SD=SD)
print('сохранено; ключи модели:', list(torch.load(f'{CKPT_DIR}/gru_ev_last.pt', map_location='cpu').keys()))

сохранено; ключи модели: ['gru.weight_ih_l0', 'gru.weight_hh_l0', 'gru.bias_ih_l0', 'gru.bias_hh_l0', 'gru.weight_ih_l1', 'gru.weight_hh_l1', 'gru.bias_ih_l1', 'gru.bias_hh_l1', 'head.weight', 'head.bias']


In [ ]:
import numpy as np, torch, os, time, shutil, zipfile, importlib.util
from types import SimpleNamespace
torch.set_num_threads(2)
SRC, OUT, NS, AX = '/content/ens3_src', '/content/ens5ev', 20, 0.07
CKS = [f'{CKPT_DIR}/gru_ev_last.pt', f'{CKPT_DIR}/gru_ev_s1_last.pt']
if not os.path.exists(f'{SRC}/weights.npz'):
    os.makedirs(SRC, exist_ok=True); zipfile.ZipFile(f'{CKPT_DIR}/ens3_submission.zip').extractall(SRC)
z_ = np.load(f'{CKPT_DIR}/evfeat_scale.npz'); MU8, SD8 = z_['MU'].astype(np.float64), z_['SD'].astype(np.float64)

def evfeat_n(x, n=20000):                                       # та же функция, что при обучении
    m_ = len(x); d = np.diff(x[:, :22], axis=0, prepend=x[:1, :22]); ch = (d != 0).any(1); ch[0] = False
    m = np.where(ch, d.mean(1), 0.0); ev = np.flatnonzero(ch); cnt = np.cumsum(ch)
    cs = np.concatenate([[0.0], np.cumsum(m[ev])]); out = np.zeros((m_, 8), np.float32); t = np.arange(m_)
    for i, k in enumerate((1, 2, 3, 5, 10)): out[:, i] = cs[cnt] - cs[np.maximum(cnt - k, 0)]
    last = ev[np.maximum(cnt - 1, 0)] if len(ev) else np.zeros(m_, int)
    out[:, 5] = np.log1p(np.where(cnt > 0, t - last, n))
    c0 = np.concatenate([[0], cnt])
    for i, w in ((6, 50), (7, 200)): out[:, i] = c0[t + 1] - c0[np.maximum(t + 1 - w, 0)]
    return out
def xin(X):                                                     # (n, L, 112) -> (n, L, 120)
    return np.stack([np.concatenate([x, ((evfeat_n(x) - MU8) / SD8).astype(np.float32)], 1) for x in X])

# ---------- модели ----------
nets = []
for ck in CKS:
    sd = torch.load(ck, map_location='cpu')
    need = {'gru.weight_ih_l0', 'gru.weight_hh_l0', 'gru.bias_ih_l0', 'gru.bias_hh_l0', 'gru.weight_ih_l1', 'gru.weight_hh_l1',
            'gru.bias_ih_l1', 'gru.bias_hh_l1', 'head.weight', 'head.bias'}
    assert need <= set(sd), f'у {ck} другие ключи: {list(sd)}'
    assert sd['gru.weight_ih_l0'].shape[1] == 120 and len(sd) == len(need), f'{ck}: не GRU 2×128 + Linear на 120 входах'
    n_ = Net2(120); n_.load_state_dict(sd); n_.eval(); nets.append((n_, sd))
def tout(net, X):
    with torch.no_grad(): return net(torch.from_numpy(xin(np.asarray(X, np.float32))))[0].numpy().astype(np.float64)

def load_pm(d, name):
    sp = importlib.util.spec_from_file_location(name, f'{d}/solution.py'); m = importlib.util.module_from_spec(sp); sp.loader.exec_module(m)
    return m.PredictionModel()
def run_pm(pm, X, sid, grab=False):
    out = np.zeros((len(X), 2), np.float32); G = np.zeros((len(X), 8), np.float32) if grab else None
    for t in range(len(X)):
        out[t] = pm.predict(SimpleNamespace(seq_ix=sid, step_in_seq=t, state=X[t], need_prediction=True))
        if grab: G[t] = pm.xe[112:]
    return (out, G) if grab else out

# ---------- статистики: старый ансамбль (3 модели) и две новые ----------
W0 = dict(np.load(f'{SRC}/weights.npz')); K_old = len(W0['mods']); assert K_old == 3, f'в ens3 {K_old} моделей'
pm_old = load_pm(SRC, 'sol_old'); Xs = np.asarray(Xv[:NS], np.float32); t0 = time.time()
old = np.stack([run_pm(pm_old, Xs[i], i) for i in range(NS)]).astype(np.float64); print(f'старый ансамбль: {time.time() - t0:.0f} с')
old_nc = old - W0['const'].astype(np.float64)
Pn = [np.concatenate([tout(n_, Xs[i:i + 5]) for i in range(0, NS, 5)]) for n_, _ in nets]
st = lambda a: (a[:, WARM:].reshape(-1, 2).mean(0), a[:, WARM:].reshape(-1, 2).std(0))
mu_o, sd_o = st(old_nc); m_old = st(old)[0]; ms = [st(p) for p in Pn]
U_ = K_old * (old_nc - mu_o) / sd_o + sum((p - m_) / s_ for p, (m_, s_) in zip(Pn, ms))
s = sd_o / st(U_)[1]; A = K_old * s / sd_o; ws = [s / s_ for _, s_ in ms]
const_new = m_old - A * mu_o - sum(w * m_ for w, (m_, _) in zip(ws, ms))

# ---------- веса: две модели типа 2 (GRU на [x, 8 событийных признаков]) ----------
mats, flat, meta, mods = W0['mats'], W0['flat'], W0['meta'], W0['mods']
for n_, sd in nets:
    mo, fo, r = len(mats), len(flat), len(meta); so = int((mods[:, 2] + 2 * mods[:, 3]).max()); mc, fc, rows = [], [], []
    def addm(a):
        global mo
        a = np.ascontiguousarray(a, dtype=np.float16); mc.append(a.view(np.uint16).reshape(-1)); rows.append([mo, a.shape[0], a.shape[1]]); mo += a.size
    def addv(v):
        global fo
        v = np.ascontiguousarray(v, dtype=np.float32); fc.append(v); rows.append([fo, 1, len(v)]); fo += len(v)
    g = lambda k: sd[k].numpy().astype(np.float32)
    addm(g('gru.weight_ih_l0').T); addm(g('gru.weight_hh_l0').T); addv(g('gru.bias_ih_l0')); addv(g('gru.bias_hh_l0'))
    addm(g('gru.weight_ih_l1').T); addm(g('gru.weight_hh_l1').T); addv(g('gru.bias_ih_l1')); addv(g('gru.bias_hh_l1'))
    addm(g('head.weight').T); addv(g('head.bias'))
    mats, flat = np.concatenate([mats] + mc), np.concatenate([flat] + fc)
    meta = np.vstack([meta, np.array(rows, np.int64)]); mods = np.vstack([mods, np.array([[2, r, so, 128]], np.int64)])
Wn = dict(W0); Wn.update(mats=mats, flat=flat, meta=meta, mods=mods)
Wn['coef'] = np.vstack([A * W0['coef']] + [w[None, :] for w in ws]).astype(np.float32); Wn['const'] = const_new.astype(np.float32)
stx = np.asarray(W0['selftest_x'], np.float32)
Wn['selftest_y'] = (A * (W0['selftest_y'].astype(np.float64) - W0['const']) + sum(w * tout(n_, stx[None])[0] for w, (n_, _) in zip(ws, nets))
                    + const_new).astype(np.float32)
Wn['ev_mu'], Wn['ev_sd'] = MU8, SD8

# ---------- solution.py: тип 2 и событийные признаки в движке ----------
EVF = '''_KS = np.array([1, 2, 3, 5, 10], dtype=np.int64)


@njit(**_KW)
def _evfeat(x, prev, first, ei, mbuf, fl, xe, emu, esd):
    """Causal event features for type-2 models (price-change events of columns 0-21); writes [x, (f - mu) / sd] into xe.
    ei: step, events so far, step of last event, events in last 50 steps, events in last 200 steps."""
    t = ei[0]
    cnt = ei[1]
    ch = False
    s = 0.0
    if not first:
        for i in range(22):
            d = x[i] - prev[i]
            if d != _ZERO:
                ch = True
            s += d
    flag = 0
    if ch:
        cnt += 1
        mbuf[cnt % 10] = s / 22.0
        ei[2] = t
        flag = 1
    j = t % 200
    ei[4] += flag - fl[j]
    ei[3] += flag - fl[(t + 150) % 200]
    fl[j] = flag
    for q in range(5):
        k = _KS[q]
        m = k if k < cnt else cnt
        acc = 0.0
        for jj in range(m):
            acc += mbuf[(cnt - jj) % 10]
        xe[112 + q] = (acc - emu[q]) / esd[q]
    v5 = math.log1p(t - ei[2]) if cnt > 0 else math.log1p(20000.0)
    xe[117] = (v5 - emu[5]) / esd[5]
    xe[118] = (ei[3] - emu[6]) / esd[6]
    xe[119] = (ei[4] - emu[7]) / esd[7]
    for i in range(112):
        xe[i] = x[i]
    ei[0] = t + 1
    ei[1] = cnt


'''
GRU0 = '_gru({}, h0, _mat(mats, meta, r), _mat(mats, meta, r + 1), _vec(flat, meta, r + 2), _vec(flat, meta, r + 3), g_i, g_h)\n'
SIG = 'def _step(x, prev, first, flat, mats, meta, mods, S, gi, gh, f, zb, cat, hid, o2, coef, const, out'
patches = [
    (f'@njit(**_KW)\n{SIG}):\n    """One row for every model; writes the blend into out and copies x into prev."""\n    out[0] = const[0]\n',
     f'{EVF}@njit(**_KW)\n{SIG}, ei, mbuf, fl, xe, emu, esd):\n    """One row for every model; writes the blend into out and copies x into prev."""\n'
     '    _evfeat(x, prev, first, ei, mbuf, fl, xe, emu, esd)\n    out[0] = const[0]\n'),
    ('        if typ == 0:\n            ' + GRU0.format('x'),
     '        if typ == 0 or typ == 2:\n            if typ == 0:\n                ' + GRU0.format('x') +
     '            else:\n                ' + GRU0.format('xe')),
    ('              self.f, self.zb, self.cat, self.hid, self.o2, self.coef, self.const, self.out)\n',
     '              self.f, self.zb, self.cat, self.hid, self.o2, self.coef, self.const, self.out,\n'
     '              self.ei, self.mbuf, self.fl, self.xe, self.emu, self.esd)\n'),
    ('        self.prev = np.zeros(112, np.float32)\n',
     '        self.prev = np.zeros(112, np.float32)\n'
     '        self.ei, self.mbuf, self.fl = np.zeros(5, np.int64), np.zeros(10, np.float64), np.zeros(200, np.int64)\n'
     '        self.xe = np.zeros(120, np.float32)\n'
     '        self.emu = np.ascontiguousarray(w["ev_mu"], dtype=np.float64) if "ev_mu" in w.files else np.zeros(8)\n'
     '        self.esd = np.ascontiguousarray(w["ev_sd"], dtype=np.float64) if "ev_sd" in w.files else np.ones(8)\n'),
    ('    def _reset(self):\n        self.S[:] = 0.0\n',
     '    def _ev_reset(self):\n        self.ei[:] = 0\n        self.mbuf[:] = 0.0\n        self.fl[:] = 0\n\n'
     '    def _reset(self):\n        self.S[:] = 0.0\n        self._ev_reset()\n'),
    ('            self.S[:] = 0.0\n            self.first = True\n        self.x[:] = data_point.state\n',
     '            self.S[:] = 0.0\n            self.first = True\n            self._ev_reset()\n        self.x[:] = data_point.state\n'),
]
src = open(f'{SRC}/solution.py').read()
assert 'перекрёстное смешивание' not in src, 'в ens3 уже есть обёртка — нужен solution.py без неё'
for old_, new_ in patches:
    assert src.count(old_) == 1, f'не нашёл место для правки: {old_[:70]!r}'
    src = src.replace(old_, new_)
src = src.replace('  1 = our GRU:', '  2 = type 0 on [raw features, 8 event features] (120 inputs), see _evfeat\n  1 = our GRU:')
src += f'''

# ---- перекрёстное смешивание таргетов, A = {AX} ----
_BasePredictionModel = PredictionModel
class PredictionModel(_BasePredictionModel):
    _C, _A = {1 - AX!r}, {AX!r}
    def predict(self, data_point):
        out = super().predict(data_point)
        if out is None: return None
        o0, o1 = float(out[0]), float(out[1])
        return np.array((self._C * o0 - self._A * o1, self._C * o1 - self._A * o0), dtype=np.float32)
'''
if os.path.exists(OUT): shutil.rmtree(OUT)
os.makedirs(OUT); np.savez(f'{OUT}/weights.npz', **Wn); open(f'{OUT}/solution.py', 'w').write(src)

# ---------- проверки ----------
pm_new = load_pm(OUT, 'sol_new'); print('самопроверка solution.py пройдена; моделей:', len(Wn['mods']))
Xe = np.asarray(Xv[20:22], np.float32)
o_new, G = run_pm(pm_new, Xe[0], 900, grab=True)
print('признаки в движке против evfeat: макс. расхождение', float(np.abs(G - (evfeat_n(Xe[0]) - MU8) / SD8).max()), '(нужно < 1e-4)')
M = np.array([[1 - AX, -AX], [-AX, 1 - AX]]); worst = 0.0
for i in range(2):
    o_old = run_pm(pm_old, Xe[i], 100 + i).astype(np.float64); o_n = run_pm(pm_new, Xe[i], 100 + i).astype(np.float64)
    exp = (A * (o_old - W0['const']) + sum(w * tout(n_, Xe[i:i + 1])[0] for w, (n_, _) in zip(ws, nets)) + const_new) @ M
    worst = max(worst, np.abs(o_n - exp).max())
print('макс. расхождение выхода с формулой:', worst, '(нужно < 1e-3)')
d1, d2 = run_pm(pm_new, Xe[0][:3000], 300), run_pm(pm_new, Xe[0][:3000], 301)
print('детерминизм:', np.array_equal(d1, d2), '| конечны:', bool(np.isfinite(d1).all()))
tm = {}
for nm, pm in (('старый', pm_old), ('новый', pm_new)):
    t0 = time.time(); run_pm(pm, Xe[0], 400); tm[nm] = time.time() - t0
print(f'оценка на платформе ≈ {26.1 * tm["новый"] / tm["старый"]:.0f} мин (от 26 мин у ens3; держимся ниже 45)')
shutil.make_archive('/content/ens5ev_submission', 'zip', OUT); shutil.copy('/content/ens5ev_submission.zip', f'{CKPT_DIR}/')
print('готово: /content/ens5ev_submission.zip (копия на Drive)')

старый ансамбль: 43 с
самопроверка solution.py пройдена; моделей: 5
признаки в движке против evfeat: макс. расхождение 1.0003139507830383e-06 (нужно < 1e-4)
макс. расхождение выхода с формулой: 0.0001716849572740431 (нужно < 1e-3)
детерминизм: True | конечны: True
оценка на платформе ≈ 44 мин (от 26 мин у ens3; держимся ниже 45)
готово: /content/ens5ev_submission.zip (копия на Drive)


In [ ]:
from google.colab import drive
import os
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
%run -i /content/drive/MyDrive/wnn/lib.py
import torch, time
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU')

D = '/content/data'
need = ['trainall_X16.npy', 'trainall_Y.npy', 'validrest_X16.npy', 'validrest_Y.npy', 'valid300_X.npy', 'valid300_Y.npy', 'valid300_M.npy']
miss = [f for f in need if not os.path.exists(f'{D}/{f}')]
assert not miss, f'нет {miss}: виртуальная машина сменилась, данные надо пересобрать ячейками 1 и 3 из инструкции к новому ноутбуку'
assert os.path.exists(f'{CKPT_DIR}/gru_all_last.pt'), 'нет gru_all_last.pt на Drive'

Xv = np.load(f'{D}/valid300_X.npy', mmap_mode='r'); Yv = np.load(f'{D}/valid300_Y.npy')
Mv = np.load(f'{D}/valid300_M.npy').astype(bool)
famv = fam_of(Yv, verbose=False); hv = np.bincount(famv, minlength=4) / len(famv)
Xva, Yva, Mva, fam_v, have = Xv, Yv, Mv, famv, hv               # для mix_score из lib.py

# predict из lib.py подаёт float16 в сеть как есть — оборачиваем, чтобы переводил во float32
class F32:
    def __init__(s, X): s.X = X
    def __len__(s): return len(s.X)
    def __getitem__(s, i): return np.asarray(s.X[i], np.float32)
if not getattr(predict, '_f32', False):
    _predict_lib = predict
    def predict(model, X, idx, *a, **k):
        if getattr(X, 'dtype', None) == np.float16: X = F32(X)
        return _predict_lib(model, X, idx, *a, **k)
    predict._f32 = True
print('окружение готово: valid300', Xv.shape, '| семейства', np.bincount(famv, minlength=4))

GPU: Tesla T4
окружение готово: valid300 (300, 20000, 112) | семейства [68 54 80 98]


In [ ]:
D = '/content/data'
XA = np.load(f'{D}/trainall_X16.npy', mmap_mode='r'); YA = np.load(f'{D}/trainall_Y.npy', mmap_mode='r')
Xva, Yva, Mva, fam_v, have = Xv, Yv, Mv, famv, hv               # обычные 112 входов, без событийных признаков

def mix_all(net, X, Y, ids):                                    # взвешенная корреляция по всем строкам после прогрева, смесь семейств valid
    Yc = np.clip(np.asarray(Y[ids], np.float64), -2, 2)[:, WARM:]; fam = fam_of(np.asarray(Y[ids]), verbose=False)
    sh = np.bincount(fam, minlength=4) / len(fam); P = predict(net, X, np.asarray(ids))[:, WARM:].astype(np.float64)
    r = []
    for j in (0, 1):
        y, p = Yc[..., j], P[..., j]; w = np.abs(y) * (VALMIX / np.maximum(sh, 1e-9))[fam][:, None]
        my, mp = (w * y).sum() / w.sum(), (w * p).sum() / w.sum()
        r.append((w * (y - my) * (p - mp)).sum() / np.sqrt((w * (y - my) ** 2).sum() * (w * (p - mp) ** 2).sum()))
    return float(np.mean(r))

net = Net().to(dev); net.load_state_dict(torch.load(f'{CKPT_DIR}/gru_all_last.pt', map_location=dev)); net.eval()
tr_ids = np.arange(0, 3000, 10)                                 # 300 последовательностей, на которых модель училась
print(f'(а) gru_all, все строки: на обучающих {mix_all(net, XA, YA, tr_ids):.4f} | на отложенных {mix_all(net, Xv, Yv, np.arange(300)):.4f}')

# ---------- (б) ещё 2 эпохи с постоянным lr 1e-4 ----------
class Cat:
    def __init__(s, A, B): s.A, s.B, s.n = A, B, len(A)
    def __len__(s): return s.n + len(s.B)
    def __getitem__(s, idx):
        idx = np.asarray(idx); a, b = idx[idx < s.n], idx[idx >= s.n] - s.n
        return np.concatenate([p for p in ((s.A[a] if len(a) else None), (s.B[b] if len(b) else None)) if p is not None]).astype(np.float32)
XT_ = Cat(XA, np.load(f'{D}/validrest_X16.npy', mmap_mode='r'))
YT_ = np.concatenate([np.asarray(YA), np.load(f'{D}/validrest_Y.npy')]); N_ = len(XT_)
print('до дообучения:', mix_score(net))
opt = torch.optim.Adam(net.parameters(), lr=1e-4); rng = np.random.default_rng(7)
for ep in range(2):
    net.train(); t0 = time.time(); run = []
    for g in range(N_ // 32):
        idx = np.sort(rng.permutation(N_)[:32]) if False else np.sort(rng.choice(N_, 32, replace=False))
        x = torch.from_numpy(XT_[idx]).to(dev); y = torch.from_numpy(YT_[idx]).to(dev).clamp(-2, 2); h = None
        for t in range(0, L, 1000):
            o, h = net(x[:, t:t + 1000], h); h = h.detach(); yy = y[:, t:t + 1000]; l = yy.abs() * (o - yy) ** 2
            if t < WARM: l = l[:, WARM - t:]
            opt.zero_grad(set_to_none=True); l.mean().backward(); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); run.append(l.mean().item())
    net.eval(); print(f'дообучение, эпоха {ep + 1}/2 | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | {mix_score(net)}', flush=True)
torch.save(net.state_dict(), f'{CKPT_DIR}/gru_all_ft.pt')

(а) gru_all, все строки: на обучающих 0.5011 | на отложенных 0.5011
до дообучения: {'все строки': 0.501, 'is_scored': 0.657}
дообучение, эпоха 1/2 | 557 с | loss 0.9033 | {'все строки': 0.501, 'is_scored': 0.656}
дообучение, эпоха 2/2 | 554 с | loss 0.9027 | {'все строки': 0.5, 'is_scored': 0.656}


In [ ]:
from google.colab import drive
import os, time
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
%run -i /content/drive/MyDrive/wnn/lib.py
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU')

D = '/content/data'
need = ['trainall_X16.npy', 'trainall_Y.npy', 'validrest_X16.npy', 'validrest_Y.npy', 'valid300_X.npy', 'valid300_Y.npy', 'valid300_M.npy']
miss = [f for f in need if not os.path.exists(f'{D}/{f}')]
assert not miss, f'нет {miss}: нужно дособрать их ячейкой 1 из инструкции к новому ноутбуку'

# ---------- valid300 и всё, что нужно mix_score ----------
Xv = np.load(f'{D}/valid300_X.npy', mmap_mode='r'); Yv = np.load(f'{D}/valid300_Y.npy')
Mv = np.load(f'{D}/valid300_M.npy').astype(bool)
famv = fam_of(Yv, verbose=False); hv = np.bincount(famv, minlength=4) / len(famv)
Xva, Yva, Mva, fam_v, have = Xv, Yv, Mv, famv, hv

# ---------- predict из lib.py: float16 → float32 ----------
class F32:
    def __init__(s, X): s.X = X
    def __len__(s): return len(s.X)
    def __getitem__(s, i): return np.asarray(s.X[i], np.float32)
if not getattr(predict, '_f32', False):
    _predict_lib = predict
    def predict(model, X, idx, *a, **k):
        if getattr(X, 'dtype', None) == np.float16: X = F32(X)
        return _predict_lib(model, X, idx, *a, **k)
    predict._f32 = True

# ---------- обучающие данные: train + 1573 valid, отдаются во float32 ----------
class Cat:
    def __init__(s, A, B): s.A, s.B, s.n = A, B, len(A)
    def __len__(s): return s.n + len(s.B)
    def __getitem__(s, idx):
        idx = np.asarray(idx); a, b = idx[idx < s.n], idx[idx >= s.n] - s.n
        return np.concatenate([p for p in ((s.A[a] if len(a) else None), (s.B[b] if len(b) else None)) if p is not None]).astype(np.float32)
XA = np.load(f'{D}/trainall_X16.npy', mmap_mode='r'); YA = np.load(f'{D}/trainall_Y.npy', mmap_mode='r')
XT_ = Cat(XA, np.load(f'{D}/validrest_X16.npy', mmap_mode='r'))
YT_ = np.concatenate([np.asarray(YA), np.load(f'{D}/validrest_Y.npy')]); N_ = len(XT_)
tr_ids = np.arange(0, 3000, 10)

def mix_all(net, X, Y, ids):                                    # корреляция по всем строкам после прогрева, смесь семейств valid
    Yc = np.clip(np.asarray(Y[ids], np.float64), -2, 2)[:, WARM:]; fam = fam_of(np.asarray(Y[ids]), verbose=False)
    sh = np.bincount(fam, minlength=4) / len(fam); P = predict(net, X, np.asarray(ids))[:, WARM:].astype(np.float64)
    r = []
    for j in (0, 1):
        y, p = Yc[..., j], P[..., j]; w = np.abs(y) * (VALMIX / np.maximum(sh, 1e-9))[fam][:, None]
        my, mp = (w * y).sum() / w.sum(), (w * p).sum() / w.sum()
        r.append((w * (y - my) * (p - mp)).sum() / np.sqrt((w * (y - my) ** 2).sum() * (w * (p - mp) ** 2).sum()))
    return float(np.mean(r))
print(f'окружение готово: обучающих {N_} (ожидаю 12180) | valid300 {Xv.shape}')

GPU: Tesla T4
окружение готово: обучающих 12180 (ожидаю 12180) | valid300 (300, 20000, 112)


In [ ]:
H_ = 256; EPOCHS_ = 4; LR_ = 1e-3; tag_ = f'gru_all_h{H_}'
torch.manual_seed(0); rng = np.random.default_rng(0)
net_w = Net(H_).to(dev)
opt = torch.optim.AdamW(net_w.parameters(), lr=LR_, weight_decay=0.01)
steps = EPOCHS_ * (N_ // 32) * (L // 1000)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR_, total_steps=steps, pct_start=0.05)
print(f'GRU 2×{H_}: {sum(p.numel() for p in net_w.parameters()) / 1e3:.0f} тыс. параметров | обучающих {N_}')
for ep in range(EPOCHS_):
    net_w.train(); t0 = time.time(); run = []; order = rng.permutation(N_)
    for g in range(N_ // 32):
        idx = np.sort(order[g * 32:(g + 1) * 32])
        x = torch.from_numpy(XT_[idx]).to(dev); y = torch.from_numpy(YT_[idx]).to(dev).clamp(-2, 2); h = None
        for t in range(0, L, 1000):
            o, h = net_w(x[:, t:t + 1000], h); h = h.detach(); yy = y[:, t:t + 1000]; l = yy.abs() * (o - yy) ** 2
            if t < WARM: l = l[:, WARM - t:]
            opt.zero_grad(set_to_none=True); l.mean().backward(); torch.nn.utils.clip_grad_norm_(net_w.parameters(), 1.0)
            opt.step(); sched.step(); run.append(l.mean().item())
        if g == 20 and ep == 0: print(f'  оценка эпохи: ~{(time.time() - t0) / 21 * (N_ // 32) / 60:.0f} мин', flush=True)
    net_w.eval(); torch.save(net_w.state_dict(), f'{CKPT_DIR}/{tag_}_ep{ep + 1}.pt')
    print(f'эпоха {ep + 1}/{EPOCHS_} | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | {mix_score(net_w)} | ориентир gru_all 0.501 / 0.657', flush=True)
print(f'(а) для широкой: на обучающих {mix_all(net_w, XA, YA, tr_ids):.4f} | на отложенных {mix_all(net_w, Xv, Yv, np.arange(300)):.4f}')

GRU 2×256: 679 тыс. параметров | обучающих 12180
  оценка эпохи: ~16 мин
эпоха 1/4 | 936 с | loss 0.9679 | {'все строки': 0.49, 'is_scored': 0.653} | ориентир gru_all 0.501 / 0.657
эпоха 2/4 | 935 с | loss 0.9353 | {'все строки': 0.498, 'is_scored': 0.657} | ориентир gru_all 0.501 / 0.657
эпоха 3/4 | 938 с | loss 0.9183 | {'все строки': 0.504, 'is_scored': 0.659} | ориентир gru_all 0.501 / 0.657
эпоха 4/4 | 942 с | loss 0.9013 | {'все строки': 0.507, 'is_scored': 0.66} | ориентир gru_all 0.501 / 0.657
(а) для широкой: на обучающих 0.5149 | на отложенных 0.5067


In [ ]:
net_w.eval(); Pw = predict(net_w, Xv, np.arange(300)).astype(np.float32); np.save(f'{CKPT_DIR}/P_gruh256_valid300.npy', Pw)
S = Mv & (np.arange(L) >= WARM)[None]; qs, qt = np.where(S)
yq = np.clip(Yv[qs, qt].astype(np.float64), -2, 2); wq = (VALMIX / hv)[famv[qs]]
def wcorr(y, p, w):
    sw = w.sum(); my, mp = (w * y).sum() / sw, (w * p).sum() / sw
    return (w * (y - my) * (p - mp)).sum() / np.sqrt((w * (y - my) ** 2).sum() * (w * (p - mp) ** 2).sum())
f3 = famv[qs] == 3
sc = lambda E: np.mean([wcorr(yq[:, j], E[:, j], np.abs(yq[:, j]) * wq) for j in (0, 1)])
sc3 = lambda E: np.mean([wcorr(yq[f3, j], E[f3, j], np.abs(yq[f3, j])) for j in (0, 1)])
P = {'base': np.clip(np.load(f'{CKPT_DIR}/Pb_valid300.npy') * 0.3, -2, 2), 'gru': np.load(f'{CKPT_DIR}/P_gru6573_valid300.npy'),
     'corr': np.load(f'{CKPT_DIR}/P_corr3k_valid300.npy'), 'all': np.load(f'{CKPT_DIR}/P_gruall_valid300.npy'),
     'ev': np.load(f'{CKPT_DIR}/P_gruev_valid300.npy'), 'ev1': np.load(f'{CKPT_DIR}/P_gruev1_valid300.npy'), 'w256': Pw}
for combo in (['all'], ['w256'], ['all', 'w256'], ['base', 'gru', 'corr'], ['base', 'gru', 'corr', 'w256'], ['base', 'w256'],
              ['base', 'ev', 'w256'], ['base', 'gru', 'corr', 'ev', 'ev1'], ['base', 'gru', 'corr', 'ev', 'ev1', 'w256']):
    E = np.mean([P[k] for k in combo], 0)[qs, qt]; print(f'{" + ".join(combo):34s} все {sc(E):.4f} | F3 {sc3(E):.4f}')

all                                все 0.6570 | F3 0.3889
w256                               все 0.6601 | F3 0.3943
all + w256                         все 0.6608 | F3 0.3943
base + gru + corr                  все 0.6610 | F3 0.4022
base + gru + corr + w256           все 0.6621 | F3 0.4020
base + w256                        все 0.6619 | F3 0.4001
base + ev + w256                   все 0.6633 | F3 0.4038
base + gru + corr + ev + ev1       все 0.6625 | F3 0.4048
base + gru + corr + ev + ev1 + w256 все 0.6629 | F3 0.4041


In [ ]:
import subprocess, onnx
cands = subprocess.run("find /content -maxdepth 7 -name 'baseline*.onnx' -not -path '*/fixsub/*' -not -path '*/liza/*' 2>/dev/null",
                       shell=True, capture_output=True, text=True).stdout.split()
good = None
for p in cands:
    m = onnx.load(p, load_external_data=False)
    dims = [d.dim_value for d in m.graph.input[0].type.tensor_type.shape.dim]; ext = sum(t.data_location == 1 for t in m.graph.initializer)
    print(f'{p}: вход {dims}, внешних весов {ext}, {os.path.getsize(p) // 1024} КБ')
    if dims[-1] == 112 and ext == 0 and good is None: good = p
assert good, 'модели организаторов (112 входов, веса внутри) не нашлось — нужна папка baseline из стартпака'
shutil.copy(good, f'{CKPT_DIR}/baseline.onnx'); print('скопировано:', good, '→', f'{CKPT_DIR}/baseline.onnx')

/content/wnn_connectome_starterpack/baseline/baseline.onnx: вход [1, 1, 112], внешних весов 0, 752 КБ
скопировано: /content/wnn_connectome_starterpack/baseline/baseline.onnx → /content/drive/MyDrive/wnn/baseline.onnx


In [ ]:
!pip install -q onnx
import onnx, zipfile, shutil, importlib.util
from onnx import numpy_helper
from types import SimpleNamespace
torch.set_num_threads(2)
OUT, AX = '/content/ens3w', 0.07
for d, z in (('/content/ens5ev', 'ens5ev_submission.zip'), ('/content/ens3_src', 'ens3_submission.zip')):
    if not os.path.exists(f'{d}/solution.py'): os.makedirs(d, exist_ok=True); zipfile.ZipFile(f'{CKPT_DIR}/{z}').extractall(d)
z_ = np.load(f'{CKPT_DIR}/evfeat_scale.npz'); MU8, SD8 = z_['MU'].astype(np.float64), z_['SD'].astype(np.float64)

def evfeat_n(x, n=20000):
    m_ = len(x); d = np.diff(x[:, :22], axis=0, prepend=x[:1, :22]); ch = (d != 0).any(1); ch[0] = False
    m = np.where(ch, d.mean(1), 0.0); ev = np.flatnonzero(ch); cnt = np.cumsum(ch)
    cs = np.concatenate([[0.0], np.cumsum(m[ev])]); out = np.zeros((m_, 8), np.float32); t = np.arange(m_)
    for i, k in enumerate((1, 2, 3, 5, 10)): out[:, i] = cs[cnt] - cs[np.maximum(cnt - k, 0)]
    last = ev[np.maximum(cnt - 1, 0)] if len(ev) else np.zeros(m_, int)
    out[:, 5] = np.log1p(np.where(cnt > 0, t - last, n)); c0 = np.concatenate([[0], cnt])
    for i, w in ((6, 50), (7, 200)): out[:, i] = c0[t + 1] - c0[np.maximum(t + 1 - w, 0)]
    return out
xin = lambda X: np.stack([np.concatenate([x, ((evfeat_n(x) - MU8) / SD8).astype(np.float32)], 1) for x in X])

# ---------- baseline организаторов: ONNX → PyTorch ----------
def onnx_to_sd(path):
    g = onnx.load(path).graph; init = {i.name: numpy_helper.to_array(i) for i in g.initializer}
    for nd in g.node:
        if nd.op_type == 'Constant':
            for a in nd.attribute:
                if a.name == 'value': init[nd.output[0]] = numpy_helper.to_array(a.t)
    grus = sorted([nd for nd in g.node if nd.op_type == 'GRU'], key=lambda nd: init[nd.input[1]].shape[2]); assert len(grus) == 2
    reo = lambda a: np.concatenate([np.split(a, 3, 0)[i] for i in (1, 0, 2)], 0); sd = {}
    for l, nd in enumerate(grus):
        W, R = init[nd.input[1]][0], init[nd.input[2]][0]; H = R.shape[1]
        B = init[nd.input[3]][0] if len(nd.input) > 3 and nd.input[3] else np.zeros(6 * H, np.float32)
        sd[f'gru.weight_ih_l{l}'], sd[f'gru.weight_hh_l{l}'] = reo(W), reo(R)
        sd[f'gru.bias_ih_l{l}'], sd[f'gru.bias_hh_l{l}'] = reo(B[:3 * H]), reo(B[3 * H:])
    W2 = [v for v in init.values() if v.dtype.kind == 'f' and v.ndim == 2 and sorted(v.shape) == [2, H]]
    b2 = [v for v in init.values() if v.dtype.kind == 'f' and v.ndim == 1 and v.shape == (2,)]; assert len(W2) == 1 and len(b2) == 1
    sd['head.weight'] = W2[0] if W2[0].shape == (2, H) else W2[0].T; sd['head.bias'] = b2[0]
    return {k: torch.from_numpy(np.ascontiguousarray(v, np.float32)) for k, v in sd.items()}

MODELS = []                                                     # (имя, state_dict, тип, коэффициент, функция прогноза)
sd_b = onnx_to_sd(f'{CKPT_DIR}/baseline.onnx'); nb = Net(128); nb.load_state_dict(sd_b); nb.eval()
Pb_ = np.load(f'{CKPT_DIR}/Pb_valid300.npy', mmap_mode='r')
with torch.no_grad(): ob, _ = nb(torch.from_numpy(np.ascontiguousarray(Xv[0:1, :3000], np.float32)))
print('перенос baseline: расхождение с ONNX', float(np.abs(ob[0].numpy()[WARM:] - Pb_[0, WARM:3000]).max()))
sd_e = torch.load(f'{CKPT_DIR}/gru_ev_last.pt', map_location='cpu'); ne = Net2(120); ne.load_state_dict(sd_e); ne.eval()
sd_w = torch.load(f'{CKPT_DIR}/gru_all_h256_ep4.pt', map_location='cpu'); nw = Net(256); nw.load_state_dict(sd_w); nw.eval()
def tor(net, X, ev=False):
    X = np.asarray(X, np.float32); X = xin(X) if ev else X
    with torch.no_grad(): return net(torch.from_numpy(X))[0].numpy().astype(np.float64)
MODELS = [('baseline', sd_b, 0, 0.3 / 3, lambda X: tor(nb, X)), ('ev', sd_e, 2, 1 / 3, lambda X: tor(ne, X, True)),
          ('w256', sd_w, 0, 1 / 3, lambda X: tor(nw, X))]

# ---------- веса для движка ----------
mats_l, flat_l, meta_l, mods_l = [], [], [], []; mo = fo = so = 0
for _, sd, typ, _, _ in MODELS:
    r = len(meta_l); H = int(sd['gru.weight_hh_l0'].shape[1])
    def addm(a):
        global mo
        a = np.ascontiguousarray(a, dtype=np.float16); mats_l.append(a.view(np.uint16).reshape(-1)); meta_l.append([mo, a.shape[0], a.shape[1]]); mo += a.size
    def addv(v):
        global fo
        v = np.ascontiguousarray(v, dtype=np.float32); flat_l.append(v); meta_l.append([fo, 1, len(v)]); fo += len(v)
    g = lambda k: sd[k].numpy().astype(np.float32)
    addm(g('gru.weight_ih_l0').T); addm(g('gru.weight_hh_l0').T); addv(g('gru.bias_ih_l0')); addv(g('gru.bias_hh_l0'))
    addm(g('gru.weight_ih_l1').T); addm(g('gru.weight_hh_l1').T); addv(g('gru.bias_ih_l1')); addv(g('gru.bias_hh_l1'))
    addm(g('head.weight').T); addv(g('head.bias'))
    mods_l.append([typ, r, so, H]); so += 2 * H
stx = np.asarray(np.load('/content/ens3_src/weights.npz')['selftest_x'], np.float32)
coef = np.array([[c, c] for _, _, _, c, _ in MODELS], np.float32)
sty = sum(c * f(stx[None])[0] for _, _, _, c, f in MODELS)
W = dict(mats=np.concatenate(mats_l), flat=np.concatenate(flat_l), meta=np.array(meta_l, np.int64), mods=np.array(mods_l, np.int64),
         coef=coef, const=np.zeros(2, np.float32), selftest_x=stx, selftest_y=sty.astype(np.float32), ev_mu=MU8, ev_sd=SD8)
if os.path.exists(OUT): shutil.rmtree(OUT)
os.makedirs(OUT); np.savez(f'{OUT}/weights.npz', **W); shutil.copy('/content/ens5ev/solution.py', f'{OUT}/solution.py')
assert 'перекрёстное смешивание' in open(f'{OUT}/solution.py').read() and '_evfeat' in open(f'{OUT}/solution.py').read()

# ---------- проверки ----------
def load_pm(d, name):
    sp = importlib.util.spec_from_file_location(name, f'{d}/solution.py'); m = importlib.util.module_from_spec(sp); sp.loader.exec_module(m)
    return m.PredictionModel()
def run_pm(pm, X, sid):
    out = np.zeros((len(X), 2), np.float32)
    for t in range(len(X)): out[t] = pm.predict(SimpleNamespace(seq_ix=sid, step_in_seq=t, state=X[t], need_prediction=True))
    return out
pm = load_pm(OUT, 'sol_3w'); print('самопроверка solution.py пройдена; моделей:', len(W['mods']))
M = np.array([[1 - AX, -AX], [-AX, 1 - AX]]); worst = 0.0
for i in (20, 21):
    X = np.asarray(Xv[i], np.float32); o = run_pm(pm, X, 100 + i).astype(np.float64)
    exp = sum(c * f(X[None])[0] for _, _, _, c, f in MODELS) @ M; worst = max(worst, np.abs(o - exp).max())
print('макс. расхождение с формулой:', worst, '(нужно < 1e-3)')
d1, d2 = run_pm(pm, np.asarray(Xv[20, :3000], np.float32), 300), run_pm(pm, np.asarray(Xv[20, :3000], np.float32), 301)
print('детерминизм:', np.array_equal(d1, d2), '| конечны:', bool(np.isfinite(d1).all()))
pm_ref = load_pm('/content/ens3_src', 'sol_ref'); X = np.asarray(Xv[22], np.float32)
t0 = time.time(); run_pm(pm_ref, X, 400); t_ref = time.time() - t0
t0 = time.time(); run_pm(pm, X, 401); t_new = time.time() - t0
print(f'оценка на платформе ≈ {26.1 * t_new / t_ref:.0f} мин (лимит 60)')
shutil.make_archive('/content/ens3w_submission', 'zip', OUT); shutil.copy('/content/ens3w_submission.zip', f'{CKPT_DIR}/')
print('готово: /content/ens3w_submission.zip (копия на Drive)')

/tmp/ipykernel_950/4068937002.py:39: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  return {k: torch.from_numpy(np.ascontiguousarray(v, np.float32)) for k, v in sd.items()}


перенос baseline: расхождение с ONNX 5.364418029785156e-06
самопроверка solution.py пройдена; моделей: 3
макс. расхождение с формулой: 0.0004533175677061685 (нужно < 1e-3)
детерминизм: True | конечны: True
оценка на платформе ≈ 32 мин (лимит 60)
готово: /content/ens3w_submission.zip (копия на Drive)


In [ ]:
NT = 8                                                          # длина ленты
def evtape(x, n=20000):
    m_ = len(x); d = np.diff(x[:, :22], axis=0, prepend=x[:1, :22]); ch = (d != 0).any(1); ch[0] = False
    ev = np.flatnonzero(ch); mev = d[ev].mean(1) if len(ev) else np.zeros(0); cnt = np.cumsum(ch); t = np.arange(m_)
    out = np.zeros((m_, 2 * NT), np.float32)
    for j in range(NT):
        i = cnt - 1 - j; ok = i >= 0; ic = np.maximum(i, 0)
        if len(ev): out[:, j] = np.where(ok, mev[ic], 0.0)                                      # сдвиг j-го с конца события
        if j == 0:
            gap = np.where(ok, t - (ev[ic] if len(ev) else 0), n)                                # шагов с последнего события
        else:
            i2 = cnt - j; ok2 = (i >= 0) & (i2 >= 0)
            gap = np.where(ok2, (ev[np.maximum(i2, 0)] - ev[ic]) if len(ev) else 0, n)           # интервал между событиями
        out[:, NT + j] = np.log1p(gap)
    return out
S_ = np.concatenate([evtape(np.asarray(XA[i], np.float32)) for i in range(0, 2000, 100)])
MUT, SDT = S_.mean(0).astype(np.float64), (S_.std(0) + 1e-6).astype(np.float64)
np.savez(f'{CKPT_DIR}/evtape_scale.npz', MU=MUT, SD=SDT); print('масштаб ленты сохранён')

class WithTape:                                                 # 112 исходных + 8 событийных + 16 ленты = 136
    def __init__(s, X): s.X = X
    def __len__(s): return len(s.X)
    def __getitem__(s, idx):
        if isinstance(idx, slice): idx = np.arange(len(s.X))[idx]
        idx = np.atleast_1d(np.asarray(idx)); b = np.asarray(s.X[idx], np.float32)
        f1 = (np.stack([evfeat_n(v) for v in b]) - MU8) / SD8; f2 = (np.stack([evtape(v) for v in b]) - MUT) / SDT
        return np.concatenate([b, f1.astype(np.float32), f2.astype(np.float32)], -1)

Xtr, Ytr = WithTape(XT_), YT_
Xva, Yva, Mva, fam_v, have = WithTape(Xv), Yv, Mv, famv, hv
print('вход:', Xtr[np.array([0])].shape[-1], '| обучающих:', len(Xtr))
torch.manual_seed(0)
net_t = train(Net2(136).to(dev), epochs=5, tag='gru_tape')
torch.save(net_t.state_dict(), f'{CKPT_DIR}/gru_tape_last.pt')
P['tape'] = predict(net_t, Xva, np.arange(300)).astype(np.float32); np.save(f'{CKPT_DIR}/P_grutape_valid300.npy', P['tape'])
Xva = Xv                                                        # вернуть обычный valid для остальных ячеек
for combo in (['ev'], ['tape'], ['base', 'ev', 'w256'], ['base', 'tape', 'w256'], ['base', 'ev', 'tape', 'w256']):
    E = np.mean([P[k] for k in combo], 0)[qs, qt]; print(f'{" + ".join(combo):26s} все {sc(E):.4f} | F3 {sc3(E):.4f}')

масштаб ленты сохранён
вход: 136 | обучающих: 12180
эпоха 1/5 | 868 с | loss 0.9750 | valid все строки 0.482 [0.48  0.484] | is_scored 0.657 [0.654 0.66 ]
эпоха 2/5 | 870 с | loss 0.9493 | valid все строки 0.490 [0.491 0.489] | is_scored 0.663 [0.659 0.667]
эпоха 3/5 | 858 с | loss 0.9369 | valid все строки 0.492 [0.493 0.49 ] | is_scored 0.666 [0.665 0.668]
эпоха 4/5 | 861 с | loss 0.9228 | valid все строки 0.501 [0.502 0.499] | is_scored 0.671 [0.667 0.675]
эпоха 5/5 | 857 с | loss 0.9095 | valid все строки 0.500 [0.501 0.5  ] | is_scored 0.672 [0.668 0.675]
ev                         все 0.6600 | F3 0.4031
tape                       все 0.6592 | F3 0.4030
base + ev + w256           все 0.6633 | F3 0.4038
base + tape + w256         все 0.6631 | F3 0.4040
base + ev + tape + w256    все 0.6637 | F3 0.4057


In [4]:
from google.colab import drive
import os, time, shutil
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
%run -i /content/drive/MyDrive/wnn/lib.py
!pip -q install onnx
import torch, onnx
from onnx import numpy_helper
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU')

D = '/content/data'
need = ['trainall_X16.npy', 'trainall_Y.npy', 'valid300_X.npy', 'valid300_Y.npy', 'valid300_M.npy']
miss = [f for f in need if not os.path.exists(f'{D}/{f}')]
assert not miss, (f'нет {miss}: виртуальная машина сменилась, данные пропали. '
                  'Нужно заново пройти ячейки 1 и 3 из инструкции к новому ноутбуку (скачивание и сборка, ~1 час)')
for f in ('corr3k_1_best.pt', 'P_corr3k_valid300.npy'): assert os.path.exists(f'{CKPT_DIR}/{f}'), f'нет {f} на Drive'

# ---------- valid300 ----------
Xv = np.load(f'{D}/valid300_X.npy', mmap_mode='r'); Yv = np.load(f'{D}/valid300_Y.npy')
Mv = np.load(f'{D}/valid300_M.npy').astype(bool)
famv = fam_of(Yv, verbose=False); hv = np.bincount(famv, minlength=4) / len(famv)
Xva, Yva, Mva, fam_v, have = Xv, Yv, Mv, famv, hv

# ---------- train во float16 ----------
XA = np.load(f'{D}/trainall_X16.npy', mmap_mode='r'); YA = np.load(f'{D}/trainall_Y.npy', mmap_mode='r')

# ---------- predict из lib.py: float16 → float32 ----------
class F32:
    def __init__(s, X): s.X = X
    def __len__(s): return len(s.X)
    def __getitem__(s, i): return np.asarray(s.X[i], np.float32)
if not getattr(predict, '_f32', False):
    _predict_lib = predict
    def predict(model, X, idx, *a, **k):
        if getattr(X, 'dtype', None) == np.float16: X = F32(X)
        return _predict_lib(model, X, idx, *a, **k)
    predict._f32 = True

# ---------- перевод baseline из ONNX в PyTorch (для второй строки проверки) ----------
if not os.path.exists(f'{CKPT_DIR}/baseline.onnx'):
    src = '/content/wnn_connectome_starterpack/baseline/baseline.onnx'
    if os.path.exists(src): shutil.copy(src, f'{CKPT_DIR}/baseline.onnx')
def onnx_to_sd(path):
    g = onnx.load(path).graph; init = {i.name: numpy_helper.to_array(i) for i in g.initializer}
    for nd in g.node:
        if nd.op_type == 'Constant':
            for a in nd.attribute:
                if a.name == 'value': init[nd.output[0]] = numpy_helper.to_array(a.t)
    grus = sorted([nd for nd in g.node if nd.op_type == 'GRU'], key=lambda nd: init[nd.input[1]].shape[2]); assert len(grus) == 2
    reo = lambda a: np.concatenate([np.split(a, 3, 0)[i] for i in (1, 0, 2)], 0); sd = {}
    for l, nd in enumerate(grus):
        W, R = init[nd.input[1]][0], init[nd.input[2]][0]; H = R.shape[1]
        B = init[nd.input[3]][0] if len(nd.input) > 3 and nd.input[3] else np.zeros(6 * H, np.float32)
        sd[f'gru.weight_ih_l{l}'], sd[f'gru.weight_hh_l{l}'] = reo(W), reo(R)
        sd[f'gru.bias_ih_l{l}'], sd[f'gru.bias_hh_l{l}'] = reo(B[:3 * H]), reo(B[3 * H:])
    W2 = [v for v in init.values() if v.dtype.kind == 'f' and v.ndim == 2 and sorted(v.shape) == [2, H]]
    b2 = [v for v in init.values() if v.dtype.kind == 'f' and v.ndim == 1 and v.shape == (2,)]; assert len(W2) == 1 and len(b2) == 1
    sd['head.weight'] = W2[0] if W2[0].shape == (2, H) else W2[0].T; sd['head.bias'] = b2[0]
    return {k: torch.from_numpy(np.ascontiguousarray(v, np.float32)) for k, v in sd.items()}
print(f'окружение готово: train {XA.shape[0]} последовательностей (нужно 10607) | valid300 {Xv.shape} | '
      f'baseline.onnx {"есть" if os.path.exists(f"{CKPT_DIR}/baseline.onnx") else "нет — строка baseline будет пропущена"}')

GPU: Tesla T4
окружение готово: train 10607 последовательностей (нужно 10607) | valid300 (300, 20000, 112) | baseline.onnx есть


In [5]:
assert len(XA) == 10607, 'нужен полный trainall (все 10 607 последовательностей)'
tr3k = np.sort(np.random.default_rng(0).choice(10607, 3000, replace=False))          # на них училась corr3k
pool = np.setdiff1d(np.arange(10607), tr3k)
hold = np.sort(np.random.default_rng(5).choice(pool, 600, replace=False))
YA_h = np.asarray(YA[hold]); famA = fam_of(YA_h, verbose=False)

def fam_scores(P, Y, fam):                                       # взвешенная корреляция по всем строкам после прогрева, по семействам
    Yc = np.clip(Y[:, WARM:].astype(np.float64), -2, 2); Pc = np.clip(P[:, WARM:].astype(np.float64), -2, 2); out = []
    for f in range(4):
        m = fam == f; r = []
        for j in (0, 1):
            y, p = Yc[m, :, j].ravel(), Pc[m, :, j].ravel(); w = np.abs(y); sw = w.sum()
            my, mp = (w * y).sum() / sw, (w * p).sum() / sw
            r.append((w * (y - my) * (p - mp)).sum() / np.sqrt((w * (y - my) ** 2).sum() * (w * (p - mp) ** 2).sum()))
        out.append(np.mean(r))
    return np.array(out)

models = {}
n3 = Net().to(dev); n3.load_state_dict(torch.load(f'{CKPT_DIR}/corr3k_1_best.pt', map_location=dev)); n3.eval()
models['corr3k (не видела ни A, ни B)'] = (predict(n3, XA, hold), np.load(f'{CKPT_DIR}/P_corr3k_valid300.npy'))
if 'onnx_to_sd' in globals() and os.path.exists(f'{CKPT_DIR}/baseline.onnx'):
    nb_ = Net().to(dev); nb_.load_state_dict({k: v.to(dev) for k, v in onnx_to_sd(f'{CKPT_DIR}/baseline.onnx').items()}); nb_.eval()
    models['baseline организаторов (мог видеть A)'] = (predict(nb_, XA, hold) * 0.3, np.load(f'{CKPT_DIR}/Pb_valid300.npy') * 0.3)

print('доли семейств:  A (train)', (np.bincount(famA, minlength=4) / len(famA)).round(3), '| B (valid)', hv.round(3))
print('последовательностей по семействам: A', np.bincount(famA, minlength=4), '| B', np.bincount(famv, minlength=4))
for name, (PA_, PB_) in models.items():
    sA, sB = fam_scores(PA_, YA_h, famA), fam_scores(PB_, Yv, famv)
    print(f'\n{name}')
    print('            ', '   '.join(f'  F{f}  ' for f in range(4)))
    print('  A (train) ', '   '.join(f'{v:.4f}' for v in sA))
    print('  B (valid) ', '   '.join(f'{v:.4f}' for v in sB))
    print('  B − A     ', '   '.join(f'{v:+.4f}' for v in sB - sA))

/tmp/ipykernel_525/3836684649.py:58: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  return {k: torch.from_numpy(np.ascontiguousarray(v, np.float32)) for k, v in sd.items()}


доли семейств:  A (train) [0.107 0.107 0.31  0.477] | B (valid) [0.227 0.18  0.267 0.327]
последовательностей по семействам: A [ 64  64 186 286] | B [68 54 80 98]

corr3k (не видела ни A, ни B)
               F0       F1       F2       F3  
  A (train)  0.5762   0.5204   0.3415   0.2807
  B (valid)  0.5619   0.4812   0.3356   0.2500
  B − A      -0.0142   -0.0392   -0.0059   -0.0307

baseline организаторов (мог видеть A)
               F0       F1       F2       F3  
  A (train)  0.4857   0.4378   0.3666   0.2736
  B (valid)  0.5707   0.4934   0.3442   0.2530
  B − A      +0.0850   +0.0555   -0.0223   -0.0206


In [7]:
XR = np.load(f'{D}/validrest_X16.npy', mmap_mode='r'); YR = np.load(f'{D}/validrest_Y.npy')
f32 = lambda X, i: np.asarray(X[np.sort(i)], np.float32)
def finetune(ck, H, mode, epochs=3, lr=3e-4, tag=None, seed=0):
    net = Net(H).to(dev); net.load_state_dict(torch.load(f'{CKPT_DIR}/{ck}', map_location=dev))
    opt = torch.optim.Adam(net.parameters(), lr=lr); rng = np.random.default_rng(seed); torch.manual_seed(seed)
    print(f'\n{ck}, режим «{mode}»: до дообучения {mix_score(net)}')
    nb = len(XR) // 32 if mode == 'valid' else len(XR) // 16
    for ep in range(epochs):
        net.train(); t0 = time.time(); order = rng.permutation(len(XR))
        for g in range(nb):
            if mode == 'valid':
                iv = np.sort(order[g * 32:(g + 1) * 32]); x = f32(XR, iv); y = YR[iv]
            else:
                iv = np.sort(order[g * 16:(g + 1) * 16]); it = np.sort(rng.choice(len(XA), 16, replace=False))
                x = np.concatenate([f32(XR, iv), f32(XA, it)]); y = np.concatenate([YR[iv], np.asarray(YA[it])])
            x = torch.from_numpy(x).to(dev); y = torch.from_numpy(y).to(dev).clamp(-2, 2); h = None
            for t in range(0, L, 1000):
                o, h = net(x[:, t:t + 1000], h); h = h.detach(); yy = y[:, t:t + 1000]; l = yy.abs() * (o - yy) ** 2
                if t < WARM: l = l[:, WARM - t:]
                opt.zero_grad(set_to_none=True); l.mean().backward(); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
        net.eval(); print(f'  эпоха {ep + 1}/{epochs} | {time.time() - t0:.0f} с | {mix_score(net)}', flush=True)
    if tag: torch.save(net.state_dict(), f'{CKPT_DIR}/{tag}.pt')
    return net

finetune('gru_all_last.pt', 128, 'valid', tag='gru_all_ftv')
finetune('gru_all_last.pt', 128, 'mix', tag='gru_all_ftmix')
finetune('gru_all_h256_ep4.pt', 256, 'valid', tag='w256_ftv')
finetune('gru_all_h256_ep4.pt', 256, 'mix', tag='w256_ftmix')


gru_all_last.pt, режим «valid»: до дообучения {'все строки': 0.501, 'is_scored': 0.657}
  эпоха 1/3 | 58 с | {'все строки': 0.501, 'is_scored': 0.656}
  эпоха 2/3 | 55 с | {'все строки': 0.499, 'is_scored': 0.652}
  эпоха 3/3 | 51 с | {'все строки': 0.497, 'is_scored': 0.65}

gru_all_last.pt, режим «mix»: до дообучения {'все строки': 0.501, 'is_scored': 0.657}
  эпоха 1/3 | 121 с | {'все строки': 0.5, 'is_scored': 0.655}
  эпоха 2/3 | 121 с | {'все строки': 0.5, 'is_scored': 0.655}
  эпоха 3/3 | 124 с | {'все строки': 0.502, 'is_scored': 0.656}

gru_all_h256_ep4.pt, режим «valid»: до дообучения {'все строки': 0.507, 'is_scored': 0.66}
  эпоха 1/3 | 126 с | {'все строки': 0.506, 'is_scored': 0.657}
  эпоха 2/3 | 125 с | {'все строки': 0.502, 'is_scored': 0.652}
  эпоха 3/3 | 121 с | {'все строки': 0.497, 'is_scored': 0.645}

gru_all_h256_ep4.pt, режим «mix»: до дообучения {'все строки': 0.507, 'is_scored': 0.66}
  эпоха 1/3 | 277 с | {'все строки': 0.503, 'is_scored': 0.655}
  эпоха 2/

Net(
  (gru): GRU(112, 256, num_layers=2, batch_first=True)
  (head): Linear(in_features=256, out_features=2, bias=True)
)

In [8]:
!pip -q install onnx onnxruntime
import onnxruntime as ort, zipfile
idx5k = np.sort(np.random.default_rng(0).choice(10607, 5000, replace=False))   # те же 5000, что у gru6573
class Sub:
    def __init__(s, X, ids): s.X, s.ids = X, ids
    def __len__(s): return len(s.ids)
    def __getitem__(s, i): return np.asarray(s.X[s.ids[np.asarray(i)]], np.float32)
Xtr, Ytr = Sub(XA, idx5k), np.asarray(YA[idx5k])
Xva, Yva, Mva, fam_v, have = Xv, Yv, Mv, famv, hv
print('обучающих:', len(Xtr), '(только train, без valid)')
torch.manual_seed(0)
net5 = train(Net().to(dev), epochs=5, tag='gru5k_trainonly')
torch.save(net5.state_dict(), f'{CKPT_DIR}/gru5k_trainonly_last.pt')
P5 = predict(net5, Xv, np.arange(300)).astype(np.float32); np.save(f'{CKPT_DIR}/P_gru5k_trainonly_valid300.npy', P5)
print('только train (5000):', mix_score(net5), '| ориентир gru6573 (5000 + 1573 valid): 0.498 / 0.657')

# ---------- одиночный сабмит в формате baseline ----------
src = '/content/wnn_connectome_starterpack/baseline/solution.py'
assert os.path.exists(src), 'нет папки baseline из стартпака'
net_c = Net(128); net_c.load_state_dict(torch.load(f'{CKPT_DIR}/gru5k_trainonly_last.pt', map_location='cpu')); net_c.eval()
os.makedirs('/content/sub_5k', exist_ok=True); z = torch.zeros(1, 1, 128)
torch.onnx.export(Step(net_c), (torch.zeros(1, 1, 112), z, z.clone()), '/content/sub_5k/model.onnx',
                  input_names=['features', 'hidden_0', 'hidden_1'], output_names=['prediction', 'hidden_0_out', 'hidden_1_out'],
                  opset_version=17, dynamo=False)
s_ = open(src).read().replace('baseline.onnx', 'model.onnx')
s_ = s_.replace('return np.asarray(prediction[0, 0], dtype=np.float32)',
                'return np.nan_to_num(np.asarray(prediction[0, 0], dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)')
assert 'nan_to_num' in s_, 'строка return в solution.py другая — пришли её'
open('/content/sub_5k/solution.py', 'w').write(s_)
sess = ort.InferenceSession('/content/sub_5k/model.onnx', providers=['CPUExecutionProvider'])
x = np.ascontiguousarray(Xv[0, :500], np.float32); h0 = np.zeros((1, 1, 128), np.float32); h1 = h0.copy(); o = []
for t in range(500):
    p, h0, h1 = sess.run(None, {'features': x[t][None, None], 'hidden_0': h0, 'hidden_1': h1}); o.append(p[0, 0])
with torch.no_grad(): ref = net_c(torch.from_numpy(x[None]))[0][0].numpy()
print('расхождение ONNX и PyTorch:', np.abs(np.array(o) - ref).max(), '(нужно < 1e-4)')
shutil.make_archive('/content/gru5k_trainonly_submission', 'zip', '/content/sub_5k')
shutil.copy('/content/gru5k_trainonly_submission.zip', f'{CKPT_DIR}/'); print('готово: /content/gru5k_trainonly_submission.zip')

обучающих: 5000 (только train, без valid)
эпоха 1/5 | 249 с | loss 1.0500 | valid все строки 0.457 [0.454 0.461] | is_scored 0.650 [0.648 0.652]
эпоха 2/5 | 246 с | loss 1.0090 | valid все строки 0.482 [0.485 0.478] | is_scored 0.661 [0.659 0.662]
эпоха 3/5 | 245 с | loss 0.9943 | valid все строки 0.489 [0.491 0.486] | is_scored 0.665 [0.661 0.67 ]
эпоха 4/5 | 244 с | loss 0.9771 | valid все строки 0.491 [0.491 0.49 ] | is_scored 0.667 [0.662 0.673]
эпоха 5/5 | 244 с | loss 0.9619 | valid все строки 0.491 [0.492 0.49 ] | is_scored 0.668 [0.665 0.671]
только train (5000): {'все строки': 0.491, 'is_scored': 0.656} | ориентир gru6573 (5000 + 1573 valid): 0.498 / 0.657


/tmp/ipykernel_525/1846531862.py:22: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(Step(net_c), (torch.zeros(1, 1, 112), z, z.clone()), '/content/sub_5k/model.onnx',
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4604: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with GRU can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


расхождение ONNX и PyTorch: 8.940697e-07 (нужно < 1e-4)
готово: /content/gru5k_trainonly_submission.zip


In [9]:
from sklearn.ensemble import HistGradientBoostingRegressor
def ev_of(x):
    ev = np.flatnonzero(np.r_[False, (np.diff(x[:, :22], axis=0) != 0).any(1)]); return ev[ev >= WARM]
def ranks(x):                                                   # номер значения среди уникальных уровней колонки (по всей последовательности)
    return np.stack([np.searchsorted(np.unique(x[:, c]), x[:, c]) for c in range(22)], 1)

# ---------- 1. похоже ли на сетку тиков ----------
for f in (3, 2):
    nu, steps, same = [], [], []
    for s in np.where(famv == f)[0][:30]:
        x = np.asarray(Xv[s, :, :22], np.float32); R = ranks(x); ev = ev_of(x)
        nu.append([len(np.unique(x[:, c])) for c in range(22)])
        dR = R[ev] - R[ev - 1]; steps.append(dR.ravel()); same.append((dR == dR[:, :1]).all(1))
    st = np.abs(np.concatenate(steps)); st = st[st > 0]
    print(f'F{f}: уникальных уровней на колонку в последовательности, медиана {int(np.median(nu))} | '
          f'|сдвиг в номерах| на событиях: 1 — {np.mean(st == 1):.2f}, 2 — {np.mean(st == 2):.2f}, ≥3 — {np.mean(st >= 3):.2f} | '
          f'все 22 колонки сдвинулись одинаково: {np.concatenate(same).mean():.2f}')

# ---------- 2. раскодирование уровня D: сырые изменения против тиков ----------
OFF = (0, 1, 2, 3, 4); rng = np.random.default_rng(0)
for f in (3, 2):
    Fr, Ft, Ds, Sq = [], [], [], []
    for s in np.where(famv == f)[0]:
        x = np.asarray(Xv[s, :, :22], np.float32); d = np.diff(x, axis=0, prepend=x[:1]); R = ranks(x).astype(np.float32)
        dR = np.diff(R, axis=0, prepend=R[:1]); ev = ev_of(x)
        y = Yv[s, ev].astype(np.float64); k = np.arange(1, len(ev) - 5); k = k[~(np.abs(y[k]) >= 2 - 1e-6).any(1)]
        Fr.append(np.concatenate([d[ev[k + o]] for o in OFF] + [x[ev[k]], x[ev[k + 3]]], 1))
        Ft.append(np.concatenate([dR[ev[k + o]] for o in OFF] + [R[ev[k + 3]] - R[ev[k]]], 1))
        Ds.append(y[k, 0] - y[k, 1]); Sq.append(np.full(len(k), s))
    Fr, Ft, T_, Sq = np.concatenate(Fr).astype(np.float32), np.concatenate(Ft).astype(np.float32), np.concatenate(Ds), np.concatenate(Sq)
    even = np.isin(Sq, np.where(famv == f)[0][::2]); res = {}
    for name, F_ in (('сырые изменения', Fr), ('сдвиги в тиках', Ft), ('оба', np.concatenate([Fr, Ft], 1))):
        P = np.zeros(len(T_)); t0 = time.time()
        for a in (True, False):
            tr, te = np.where(even == a)[0], np.where(even != a)[0]; sub = rng.choice(tr, min(200_000, len(tr)), replace=False)
            m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, max_leaf_nodes=63, early_stopping=False, random_state=0)
            P[te] = m.fit(F_[sub], T_[sub]).predict(F_[te])
        res[name] = np.corrcoef(P, T_)[0, 1]; print(f'  F{f}, {name:16s}: corr с уровнем D = {res[name]:.3f} ({time.time() - t0:.0f} с)', flush=True)

F3: уникальных уровней на колонку в последовательности, медиана 8 | |сдвиг в номерах| на событиях: 1 — 0.35, 2 — 0.12, ≥3 — 0.53 | все 22 колонки сдвинулись одинаково: 0.00
F2: уникальных уровней на колонку в последовательности, медиана 22 | |сдвиг в номерах| на событиях: 1 — 0.28, 2 — 0.12, ≥3 — 0.60 | все 22 колонки сдвинулись одинаково: 0.00
  F3, сырые изменения : corr с уровнем D = 0.873 (130 с)
  F3, сдвиги в тиках  : corr с уровнем D = 0.860 (106 с)
  F3, оба             : corr с уровнем D = 0.868 (230 с)
  F2, сырые изменения : corr с уровнем D = 0.697 (50 с)
  F2, сдвиги в тиках  : corr с уровнем D = 0.683 (37 с)
  F2, оба             : corr с уровнем D = 0.689 (80 с)


In [10]:
from scipy.ndimage import uniform_filter1d
famT = np.concatenate([fam_of(np.asarray(YA[i:i + 500]), verbose=False) for i in range(0, len(YA), 500)])
rate = np.array([(np.diff(np.asarray(XA[s, ::4, :22], np.float32), axis=0) != 0).any(1).mean() for s in range(len(XA))])
print('доли семейств по десятым частям train (в порядке номеров последовательностей):')
for q in range(10):
    a, b = q * len(famT) // 10, (q + 1) * len(famT) // 10
    print(f'  {q + 1:>2}/10: ' + ' '.join(f'F{f} {np.mean(famT[a:b] == f):.3f}' for f in range(4)) + f' | частота ценовых событий {rate[a:b].mean():.3f}')
print('valid (для сравнения): ' + ' '.join(f'F{f} {VALMIX[f]:.3f}' for f in range(4)))
for name, v in (('семейство F0', (famT == 0).astype(float)), ('частота событий', rate)):
    v = v - v.mean(); ac = [float((v[:-l] * v[l:]).mean() / v.var()) for l in (1, 10, 100)]
    print(f'автокорреляция по номеру последовательности, {name}: лаги 1/10/100 → {np.round(ac, 3)}')

доли семейств по десятым частям train (в порядке номеров последовательностей):
   1/10: F0 0.106 F1 0.123 F2 0.339 F3 0.433 | частота ценовых событий 0.282
   2/10: F0 0.136 F1 0.105 F2 0.336 F3 0.423 | частота ценовых событий 0.291
   3/10: F0 0.113 F1 0.112 F2 0.351 F3 0.424 | частота ценовых событий 0.271
   4/10: F0 0.140 F1 0.112 F2 0.318 F3 0.430 | частота ценовых событий 0.282
   5/10: F0 0.115 F1 0.123 F2 0.332 F3 0.430 | частота ценовых событий 0.284
   6/10: F0 0.122 F1 0.120 F2 0.320 F3 0.439 | частота ценовых событий 0.284
   7/10: F0 0.116 F1 0.111 F2 0.315 F3 0.458 | частота ценовых событий 0.283
   8/10: F0 0.120 F1 0.122 F2 0.314 F3 0.445 | частота ценовых событий 0.284
   9/10: F0 0.133 F1 0.136 F2 0.333 F3 0.399 | частота ценовых событий 0.290
  10/10: F0 0.131 F1 0.122 F2 0.332 F3 0.416 | частота ценовых событий 0.290
valid (для сравнения): F0 0.212 F1 0.185 F2 0.230 F3 0.372
автокорреляция по номеру последовательности, семейство F0: лаги 1/10/100 → [ 0.     0.008 -0

In [12]:
from scipy.signal import lfilter
XR = np.load(f'{D}/validrest_X16.npy', mmap_mode='r'); YR = np.load(f'{D}/validrest_Y.npy')
N_TR, EP = 3000, 3
ids = np.sort(np.random.default_rng(0).choice(10607, N_TR, replace=False)); YTr = np.asarray(YA[ids]); famTr = famT[ids]
z_ = np.load(f'{CKPT_DIR}/evfeat_scale.npz'); MU8, SD8 = z_['MU'], z_['SD']
def evfeat_n(x, n=20000):
    m_ = len(x); d = np.diff(x[:, :22], axis=0, prepend=x[:1, :22]); ch = (d != 0).any(1); ch[0] = False
    m = np.where(ch, d.mean(1), 0.0); ev = np.flatnonzero(ch); cnt = np.cumsum(ch)
    cs = np.concatenate([[0.0], np.cumsum(m[ev])]); out = np.zeros((m_, 8), np.float32); t = np.arange(m_)
    for i, k in enumerate((1, 2, 3, 5, 10)): out[:, i] = cs[cnt] - cs[np.maximum(cnt - k, 0)]
    last = ev[np.maximum(cnt - 1, 0)] if len(ev) else np.zeros(m_, int)
    out[:, 5] = np.log1p(np.where(cnt > 0, t - last, n)); c0 = np.concatenate([[0], cnt])
    for i, w in ((6, 50), (7, 200)): out[:, i] = c0[t + 1] - c0[np.maximum(t + 1 - w, 0)]
    return out
def f_ev(x): return ((evfeat_n(x) - MU8) / SD8).astype(np.float32)
def f_ema(x, a=1 / 500):                                         # вход минус его причинное скользящее среднее
    e = lfilter([a], [1, a - 1], x - x[:1], axis=0) + x[:1]; return np.clip(x - e, -10, 10).astype(np.float32)
FEAT = {'ev': (f_ev, 8), 'ema': (f_ema, 112)}
def getx(X, i, feats):
    b = np.asarray(X[np.atleast_1d(i)], np.float32)
    return b if not feats else np.concatenate([b] + [np.stack([FEAT[f][0](v) for v in b]) for f in feats], -1)

class G(nn.Module):
    def __init__(s, nin, H, drop): super().__init__(); s.gru = nn.GRU(nin, H, 2, batch_first=True, dropout=drop); s.head = nn.Linear(H, 2)
    def forward(s, x, h=None): y, h = s.gru(x, h); return s.head(y), h
def wpear(p, y, w):
    sw = w.sum() + 1e-8; my, mp = (w * y).sum() / sw, (w * p).sum() / sw
    return ((w * (y - my) * (p - mp)).sum() / sw) / torch.sqrt(((w * (y - my) ** 2).sum() / sw) * ((w * (p - mp) ** 2).sum() / sw) + 1e-12)

@torch.no_grad()
def pred(net, X, ids_, feats, bs=50):
    net.eval(); out = []
    for s in range(0, len(ids_), bs):
        x = torch.from_numpy(getx(X, ids_[s:s + bs], feats)).to(dev); h = None; o_ = []
        for t in range(0, L, 2000): o, h = net(x[:, t:t + 2000], h); o_.append(o)
        out.append(torch.cat(o_, 1).clamp(-2, 2).cpu().numpy())
    return np.concatenate(out)
def sums(P, Y, M):
    T = np.zeros((2, 6))
    for j in (0, 1):
        y = np.clip(Y[..., j][M], -2, 2).astype(np.float64); p = P[..., j][M].astype(np.float64); a = np.abs(y)
        T[j] = [a.sum(), (a * y).sum(), (a * p).sum(), (a * y * y).sum(), (a * p * p).sum(), (a * y * p).sum()]
    return T
def corrT(T):
    sw = T[:, 0]; my, mp = T[:, 1] / sw, T[:, 2] / sw
    return float(np.mean((T[:, 5] / sw - my * mp) / np.sqrt((T[:, 3] / sw - my ** 2) * (T[:, 4] / sw - mp ** 2))))
post = np.arange(L) >= WARM; RES2 = {}

def run2(name, H=128, loss='wmse', feats=(), drop=0.0, wd=0.0, lr=2e-3, reweight=False, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed); nin = 112 + sum(FEAT[f][1] for f in feats)
    net = G(nin, H, drop).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd) if wd > 0 else torch.optim.Adam(net.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=EP * (N_TR // 32) * (L // 1000), pct_start=0.05)
    trmix = np.bincount(famTr, minlength=4) / N_TR; wf = VALMIX / trmix; wseq = wf[famTr] / wf[famTr].mean()
    t0 = time.time()
    for ep in range(EP):
        net.train(); order = rng.permutation(N_TR)
        for g in range(N_TR // 32):
            k = np.sort(order[g * 32:(g + 1) * 32]); x = torch.from_numpy(getx(XA, ids[k], feats)).to(dev)
            y = torch.from_numpy(YTr[k]).to(dev).clamp(-2, 2); ws = torch.from_numpy((wseq[k] if reweight else np.ones(32)).astype(np.float32)).to(dev)
            h = None
            for t in range(0, L, 1000):
                o, h = net(x[:, t:t + 1000], h); h = h.detach(); yy = y[:, t:t + 1000]
                m = (torch.arange(t, t + 1000, device=dev) >= WARM)[None].expand(len(k), -1); w3 = ws[:, None, None] * yy.abs()
                l = (w3 * (o - yy) ** 2)[m[..., None].expand_as(yy)].mean()
                if loss == 'pearson2+wmse':
                    l = l + 1 - 0.5 * sum(wpear(o[..., j][m], yy[..., j][m], w3[..., j][m]) for j in (0, 1))
                opt.zero_grad(set_to_none=True); l.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step()
    P3 = pred(net, Xv, np.arange(300), feats); Ma = np.broadcast_to(post, (300, L))
    Ssc = sums(P3, Yv, Mv & Ma); Sall = sums(P3, Yv, Ma); f3 = famv == 3
    S3 = sums(P3[f3], Yv[f3], (Mv & Ma)[f3])
    for s in range(0, len(XR), 300):
        i = np.arange(s, min(s + 300, len(XR))); PR = pred(net, XR, i, feats); Sall += sums(PR, YR[i], np.broadcast_to(post, (len(i), L)))
    r = {'is_scored (300)': corrT(Ssc), 'F3 is_scored': corrT(S3), 'все строки (1873)': corrT(Sall), 'минут': (time.time() - t0) / 60}
    RES2[name] = r; json.dump(RES2, open(f'{CKPT_DIR}/results_train_to_valid.json', 'w'), ensure_ascii=False, indent=1)
    print(f'{name:34s} is_scored {r["is_scored (300)"]:.4f} | F3 {r["F3 is_scored"]:.4f} | все строки (1873) {r["все строки (1873)"]:.4f} | {r["минут"]:.1f} мин', flush=True)

import json
for s in range(3): run2(f'база, сид {s}', seed=s)
run2('Пирсон + wmse', loss='pearson2+wmse')
run2('перевзвешивание семейств под valid', reweight=True)
run2('GRU 2×256 (AdamW, lr 1e-3)', H=256, lr=1e-3, wd=0.01)
run2('событийные признаки', feats=('ev',))
run2('регуляризация (dropout 0.2, wd 1e-4)', drop=0.2, wd=1e-4)
run2('вход + (вход минус среднее за 500)', feats=('ema',))

база, сид 0                        is_scored 0.6670 | F3 0.3981 | все строки (1873) 0.4923 | 8.4 мин
база, сид 1                        is_scored 0.6656 | F3 0.3952 | все строки (1873) 0.4916 | 7.9 мин
база, сид 2                        is_scored 0.6673 | F3 0.4005 | все строки (1873) 0.4910 | 7.7 мин
Пирсон + wmse                      is_scored 0.6677 | F3 0.3993 | все строки (1873) 0.4924 | 7.9 мин
перевзвешивание семейств под valid is_scored 0.6681 | F3 0.3986 | все строки (1873) 0.4937 | 7.8 мин
GRU 2×256 (AdamW, lr 1e-3)         is_scored 0.6667 | F3 0.3919 | все строки (1873) 0.4940 | 14.2 мин
событийные признаки                is_scored 0.6662 | F3 0.3944 | все строки (1873) 0.4924 | 9.6 мин
регуляризация (dropout 0.2, wd 1e-4) is_scored 0.6653 | F3 0.3946 | все строки (1873) 0.4922 | 8.0 мин
вход + (вход минус среднее за 500) is_scored 0.6671 | F3 0.3976 | все строки (1873) 0.4934 | 19.4 мин


In [13]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def seqstats(x):                                                 # характеристики последовательности: среднее и разброс колонок, частота событий
    x = np.asarray(x[::10], np.float32); ev = (np.diff(x[:, :22], axis=0) != 0).any(1).mean()
    return np.concatenate([x.mean(0), x.std(0), [ev]])
t0 = time.time()
ST = np.stack([seqstats(XA[s]) for s in range(len(XA))] + [seqstats(Xv[s]) for s in range(300)] + [seqstats(XR[s]) for s in range(len(XR))])
famR = fam_of(YR, verbose=False); FAM = np.concatenate([famT, famv, famR])
isv = np.r_[np.zeros(len(XA)), np.ones(300 + len(XR))].astype(int)
F_ = np.column_stack([ST, np.eye(4)[FAM]]); print(f'характеристики: {F_.shape}, {time.time() - t0:.0f} с')

# ---------- насколько train и valid различимы ----------
oof = np.zeros(len(F_))
for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(F_, isv * 4 + FAM):
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, random_state=0).fit(F_[tr], isv[tr])
    oof[te] = m.predict_proba(F_[te])[:, 1]
print(f'\nAUC «train против valid»: в целом {roc_auc_score(isv, oof):.3f}')
for f in range(4):
    mf = FAM == f; print(f'  внутри F{f}: {roc_auc_score(isv[mf], oof[mf]):.3f}  (train {int((mf & (isv == 0)).sum())}, valid {int((mf & (isv == 1)).sum())})')
names = [f'среднее col{c}' for c in range(112)] + [f'разброс col{c}' for c in range(112)] + ['частота событий']
ua = []
for c in range(ST.shape[1]):
    a = np.mean([roc_auc_score(isv[FAM == f], ST[FAM == f, c]) for f in range(4)]); ua.append(abs(a - 0.5))
o = np.argsort(ua)[::-1][:12]
print('самые сдвинутые характеристики (средний |AUC − 0.5| внутри семейств):', [(names[i], round(float(ua[i]), 3)) for i in o])

# ---------- обучение на 3000 последовательностях train, «похожих на valid» ----------
p = np.clip(oof[:len(XA)], 1e-3, 1 - 1e-3); w = p / (1 - p); w = np.clip(w / w.mean(), 0.1, 10)
print(f'\nвеса последовательностей train: эффективный размер {w.sum() ** 2 / (w ** 2).sum():.0f} из {len(w)} | доли семейств среди 3000 отобранных:', end=' ')
ids_keep, YTr_keep, famTr_keep = ids, YTr, famTr
ids = np.sort(np.random.default_rng(0).choice(len(XA), N_TR, replace=False, p=w / w.sum()))
YTr, famTr = np.asarray(YA[ids]), famT[ids]
print((np.bincount(famTr, minlength=4) / N_TR).round(3), '| у valid', VALMIX.round(3))
run2('отбор train, похожих на valid')
ids, YTr, famTr = ids_keep, YTr_keep, famTr_keep
print('ориентиры: база 0.6666 / 0.4916, перевзвешивание семейств 0.6681 / 0.4937')

характеристики: (12480, 229), 224 с

AUC «train против valid»: в целом 0.995
  внутри F0: 0.994  (train 1305, valid 398)
  внутри F1: 0.999  (train 1257, valid 347)
  внутри F2: 0.994  (train 3488, valid 431)
  внутри F3: 0.993  (train 4557, valid 697)
самые сдвинутые характеристики (средний |AUC − 0.5| внутри семейств): [('среднее col78', 0.243), ('среднее col92', 0.235), ('среднее col40', 0.226), ('среднее col87', 0.201), ('среднее col25', 0.198), ('разброс col106', 0.198), ('среднее col91', 0.191), ('разброс col44', 0.19), ('среднее col39', 0.186), ('среднее col86', 0.182), ('среднее col43', 0.173), ('среднее col26', 0.173)]

веса последовательностей train: эффективный размер 682 из 10607 | доли семейств среди 3000 отобранных: [0.126 0.11  0.324 0.44 ] | у valid [0.212 0.185 0.23  0.372]
отбор train, похожих на valid      is_scored 0.6645 | F3 0.3900 | все строки (1873) 0.4931 | 8.1 мин
ориентиры: база 0.6666 / 0.4916, перевзвешивание семейств 0.6681 / 0.4937


In [ ]:
V = np.r_[22:44, 74:96]                                          # объёмы обоих инструментов
def _ema(x, a=1 / 500):
    return lfilter([a], [1, a - 1], x - x[:1], axis=0) + x[:1]
def t_vol(x):  y = x.copy(); y[:, V] = np.clip(x[:, V] - _ema(x[:, V]), -10, 10); return y
def t_all(x):  return np.clip(x - _ema(x), -10, 10).astype(np.float32)
def t_volz(x):
    y = x.copy(); m = _ema(x[:, V]); v = _ema((x[:, V] - m) ** 2); y[:, V] = np.clip((x[:, V] - m) / np.sqrt(v + 1e-2), -10, 10); return y
TRANS = {'vol': t_vol, 'all': t_all, 'volz': t_volz}

def getx(X, i, feats):                                           # «!имя» в feats = заменить вход преобразованием
    b = np.asarray(X[np.atleast_1d(i)], np.float32)
    for f in feats:
        if f.startswith('!'): b = np.stack([TRANS[f[1:]](v) for v in b]).astype(np.float32)
    add = [np.stack([FEAT[f][0](v) for v in b]) for f in feats if not f.startswith('!')]
    return np.concatenate([b] + add, -1) if add else b
for k in TRANS: FEAT['!' + k] = (None, 0)

run2('объёмы → отклонение от среднего', feats=('!vol',))
run2('все колонки → отклонение от среднего', feats=('!all',))
run2('объёмы → скользящий z-score', feats=('!volz',))
print('ориентиры: база 0.6666 / 0.4916 (разброс сидов 0.0017 / 0.0013)')

объёмы → отклонение от среднего    is_scored 0.6631 | F3 0.3842 | все строки (1873) 0.4932 | 14.3 мин
все колонки → отклонение от среднего is_scored 0.6658 | F3 0.3964 | все строки (1873) 0.4893 | 18.5 мин
